In [ ]:
!pip install nba_api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 27.7 MB/s eta 0:00:00


In [ ]:
from nba_api.stats.endpoints import PlayerGameLogs
from nba_api.stats.endpoints import PlayerGameLog
from nba_api.stats.endpoints import PlayerCareerStats
from nba_api.stats.static import players
from nba_api.stats.endpoints import CommonPlayerInfo
from nba_api.stats.endpoints import TeamGameLog
from nba_api.stats.library.parameters import SeasonAll
from nba_api.stats.endpoints import LeagueGameLog

import numpy as np
import pandas as pd

from scipy.stats import truncnorm

from IPython.display import display
pd.set_option('display.max_columns', None)

import time
import json
import stat_utils

In [ ]:
MAX_LINES = 35

def generate_parlays_batch(
  avg, cumm_avg, matchup_avg, std_last5, std_cum, last10_mat,
  gp_against_team=None,
  k=1.0, step=0.5,
  matchup_weight_cap=0.25, matchup_games_scale=8.0,
  recent_weight=0.35,
  min_sigma_floor=0.5
):
  # has_matchup = ~np.isnan(matchup_avg)

  # anchor = np.where(
  #   has_matchup,
  #   0.20 * avg + 0.35 * cumm_avg + 0.45 * matchup_avg,  # Original formula
  #   0.55 * cumm_avg + 0.45 * avg  # Fallback (no matchup data)
  # )
  # sigma = 0.7 * std_last5 + 0.3 * std_cum
  # spread = k * sigma

  has_matchup = np.isfinite(matchup_avg)

  if gp_against_team is None:
    gp_against_team = np.zeros_like(avg)

  gp = np.nan_to_num(gp_against_team, nan=0.0)
  matchup_w = np.minimum(gp / matchup_games_scale, matchup_weight_cap)
  recent_w = recent_weight
  season_w = 1.0 - recent_w - matchup_w

  anchor = np.where(
    has_matchup,
    recent_w * avg + season_w * cumm_avg + matchup_w * matchup_avg,
    # fallback: no matchup info
    0.35 * avg + 0.65 * cumm_avg
  )

  sigma_raw = 0.7 * std_last5 + 0.3 * std_cum
  sigma_fallback = np.where(np.isfinite(std_last5), std_last5, std_cum)
  sigma = np.where(np.isfinite(sigma_raw), sigma_raw, sigma_fallback)
  sigma = np.where(np.isfinite(sigma), sigma, min_sigma_floor)
  sigma = np.maximum(sigma, min_sigma_floor)

  spread = k * sigma

  half = MAX_LINES // 2
  offsets = np.arange(-half, half + 1) * step
  offsets = offsets[:MAX_LINES]

  parlays = anchor[:, None] + offsets[None, :]

  mask = np.abs(offsets)[None, :] <= spread[:, None]
  parlays = np.where(mask, parlays, np.nan)

  parlays = np.round(parlays * 2) / 2
  parlays = np.clip(parlays, 0.5, None)

  for i in range(parlays.shape[0]):
    row = parlays[i]
    valid_vals = row[np.isfinite(row)]
    if valid_vals.size == 0:
      continue

    uniq = np.unique(valid_vals)
    uniq = uniq[np.argsort(np.abs(uniq - anchor[i]))]
    uniq = uniq[:MAX_LINES]

    new_row = np.full(MAX_LINES, np.nan)
    new_row[:len(uniq)] = uniq
    parlays[i] = new_row

  # over10_rate = np.nanmean(last10_mat[:, :, None] > parlays[:, None, :], axis=1)
  # over5_rate  = np.nanmean(last10_mat[:, -5:, None] > parlays[:, None, :], axis=1)

  return parlays, anchor

def sanitize_parlay_matrix(mat, width):
  """
  Defensive version: returns an all-NaN (N,width) matrix if input is malformed.
  """
  if mat is None or not hasattr(mat, "ndim") or mat.ndim != 2 or mat.shape[1] != width:
    n = 0 if mat is None else mat.shape[0]
    return np.full((n, width), np.nan)

  bad_rows = ~np.isfinite(mat).any(axis=1)
  mat = mat.copy()
  mat[bad_rows] = np.nan
  return mat

# Builds a matrix for last 10 games as an array for every row
def build_last10_matrix(values):
  n = len(values)
  out = np.full((n, 10), np.nan)
  if n == 0:
    return out
  idx = np.arange(n)[:, None] + np.arange(-10, 0)[None, :]
  valid = idx >= 0
  out[valid] = values[idx[valid]]
  return out

# Counts the number of games in last n days
def count_games_last_days(series, days):
  dates = series.values
  window = pd.Timedelta(days=days)

  result = np.zeros(len(dates), dtype=int)
  left = 0

  for right in range(len(dates)):
    while left < right and dates[left] < dates[right] - window:
      left += 1
    result[right] = right - left

  return result

def _grouped_rolling_from_shifted(shifted, keys, window, min_periods=1, agg="mean", ddof=0):
  """
  Fast helper:
  - `shifted` should already be group-shifted
  - `keys` is a list like [df['PLAYER_ID'], df['MATCHUP']]
  - avoids lambda/apply per group
  """
  gb = shifted.groupby(keys, sort=False)
  if agg == "mean":
    out = gb.rolling(window=window, min_periods=min_periods).mean()
  elif agg == "std":
    out = gb.rolling(window=window, min_periods=min_periods).std(ddof=ddof)
  elif agg == "sum":
    out = gb.rolling(window=window, min_periods=min_periods).sum()
  else:
    raise ValueError(f"Unsupported agg={agg}")

  return out.reset_index(level=list(range(len(keys))), drop=True)

def _grouped_expanding_from_shifted(shifted, keys, agg="mean", ddof=0):
  """
  Fast helper for grouped expanding aggregations on already-shifted data.
  """
  gb = shifted.groupby(keys, sort=False)
  if agg == "mean":
    out = gb.expanding().mean()
  elif agg == "std":
    out = gb.expanding().std(ddof=ddof)
  else:
    raise ValueError(f"Unsupported agg={agg}")

  return out.reset_index(level=list(range(len(keys))), drop=True)

# Ensures that parlay is (N, width), Any row that is all-Nan or wrong shape becomes all-NaN row
def build_matchup_engine(seasons):
  all_matchups = []

  for season in seasons:
    print(f"Fetching Team Logs for {season}...")
    try:
      df_inseason = LeagueGameLog(player_or_team_abbreviation="T", season=season,season_type_all_star="Regular Season").get_data_frames()[0]
      df_playoffs = LeagueGameLog(player_or_team_abbreviation="T",season=season,season_type_all_star="Playoffs").get_data_frames()[0]

      full_log = pd.concat([df_inseason, df_playoffs], ignore_index=True)
      full_log['GAME_DATE'] = pd.to_datetime(full_log['GAME_DATE'])
      full_log = full_log.sort_values('GAME_DATE')

      full_log['POSS'] = (
        full_log['FGA'] +
        0.44 * full_log['FTA'] -
        full_log['OREB'] +
        full_log['TOV']
      )

      opp_stats = full_log[[
        'GAME_ID', 'TEAM_ID', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV',
        'FG3M', 'FGM', 'FGA', 'FG3A', 'FTM', 'FTA', 'POSS', 'OREB'
      ]].copy()

      opp_stats.columns = [
        'GAME_ID', 'OPP_TEAM_ID', 'PTS_ALLOWED', 'REB_ALLOWED', 'AST_ALLOWED',
        'STL_ALLOWED', 'BLK_ALLOWED', 'TOV_ALLOWED', '3PM_ALLOWED', 'FGM_ALLOWED',
        'FGA_ALLOWED', '3PA_ALLOWED', 'FTM_ALLOWED', 'FTA_ALLOWED', 'OPP_POSS', 'OREB_ALLOWED'
      ]

      merged = full_log.merge(opp_stats, on='GAME_ID')
      merged = merged[merged['TEAM_ID'] != merged['OPP_TEAM_ID']]

      # 5. Advanced Game-Level Metrics
      merged['GAME_PACE'] = 48 * ((merged['POSS'] + merged['OPP_POSS']) /(2 * (merged['MIN'] / 5).replace(0, np.nan)))
      merged['GAME_DEF_RATING'] = 100 * (merged['PTS_ALLOWED'] / merged['OPP_POSS'].replace(0, np.nan))

      # 6. Calculate ROLLING and SEASON Opponent Allowed Stats
      merged = merged.sort_values(['OPP_TEAM_ID', 'GAME_DATE'])

      metrics_map = {
        'PTS': 'PTS',
        'REB': 'REB',
        'AST': 'AST',
        'STL': 'STL',
        'BLK': 'BLK',
        'TOV': 'TOV',
        'FG3M': '3PM',
        'FGM': 'FGM',
        'FGA': 'FGA',
        'FG3A': '3PA',
        'FTM': 'FTM',
        'FTA': 'FTA',
        'GAME_PACE': 'PACE',
        'GAME_DEF_RATING': 'DEF_RATING'
      }

      raw_cols = list(metrics_map.keys())
      opp_key = [merged['OPP_TEAM_ID']]

      shifted_opp = merged.groupby('OPP_TEAM_ID', sort=False)[raw_cols].shift(1)
      l5_opp = _grouped_rolling_from_shifted(shifted=shifted_opp,keys=opp_key,window=5,min_periods=1,agg="mean")
      cum_opp = _grouped_expanding_from_shifted(shifted=shifted_opp,keys=opp_key,agg="mean")

      l5_opp.columns = [f"OPP_ALLOWED_{metrics_map[c]}_L5_AVG" for c in raw_cols]
      cum_opp.columns = [f"OPP_ALLOWED_{metrics_map[c]}_CUM_AVG" for c in raw_cols]

      # Rename PACE and DEF_RATING for consistency (remove "ALLOWED" prefix)
      merged = pd.concat([merged, l5_opp, cum_opp], axis=1)
      merged = merged.rename(columns={
        "OPP_ALLOWED_DEF_RATING_L5_AVG": "OPP_DEF_RATING_L5_AVG",
        "OPP_ALLOWED_DEF_RATING_CUM_AVG": "OPP_DEF_RATING_CUM_AVG",
        "OPP_ALLOWED_PACE_L5_AVG": "OPP_PACE_L5_AVG",
        "OPP_ALLOWED_PACE_CUM_AVG": "OPP_PACE_CUM_AVG",
      })

      # Select columns
      cols_to_keep = [
        'GAME_ID', 'TEAM_ABBREVIATION',
        'OPP_ALLOWED_PTS_L5_AVG', 'OPP_ALLOWED_PTS_CUM_AVG',
        'OPP_ALLOWED_REB_L5_AVG', 'OPP_ALLOWED_REB_CUM_AVG',
        'OPP_ALLOWED_AST_L5_AVG', 'OPP_ALLOWED_AST_CUM_AVG',
        'OPP_ALLOWED_STL_L5_AVG', 'OPP_ALLOWED_STL_CUM_AVG',
        'OPP_ALLOWED_BLK_L5_AVG', 'OPP_ALLOWED_BLK_CUM_AVG',
        'OPP_ALLOWED_TOV_L5_AVG', 'OPP_ALLOWED_TOV_CUM_AVG',
        'OPP_ALLOWED_3PM_L5_AVG', 'OPP_ALLOWED_3PM_CUM_AVG',
        'OPP_ALLOWED_FGM_L5_AVG', 'OPP_ALLOWED_FGM_CUM_AVG',
        'OPP_ALLOWED_FGA_L5_AVG', 'OPP_ALLOWED_FGA_CUM_AVG',
        'OPP_ALLOWED_3PA_L5_AVG', 'OPP_ALLOWED_3PA_CUM_AVG',
        'OPP_ALLOWED_FTM_L5_AVG', 'OPP_ALLOWED_FTM_CUM_AVG',
        'OPP_ALLOWED_FTA_L5_AVG', 'OPP_ALLOWED_FTA_CUM_AVG',
        'OPP_PACE_L5_AVG', 'OPP_PACE_CUM_AVG',
        'OPP_DEF_RATING_L5_AVG', 'OPP_DEF_RATING_CUM_AVG',
      ]

      merged = merged[cols_to_keep]
      all_matchups.append(merged)
      time.sleep(1.5)

    except Exception as e:
      print(f"Error fetching season {season}: {e}")
      import traceback
      traceback.print_exc()
      continue

  final_df = pd.concat(all_matchups, ignore_index=True)
  print(f"\n✅ Final matchup_engine shape: {final_df.shape}")
  print(f"Final columns: {final_df.columns.tolist()}")
  return final_df

In [ ]:
STAT_COLS = stat_utils.STAT_COLS
ADVANCED_COLS = stat_utils.ADVANCED_COLS
MATCHUP_ALLOWED_METRICS_W_PACE_DEF = stat_utils.MATCHUP_ALLOWED_METRICS_W_PACE_DEF
MATCHUP_ALLOWED_METRICS = stat_utils.MATCHUP_ALLOWED_METRICS

# Function to create dataframe as well as add on to existing dataframe
def create_df(df: pd.DataFrame, players, seasons, matchup_df: pd.DataFrame):

  regular_season_ids = [f"2{s[:4]}" for s in seasons]
  playoff_season_ids = [f"4{s[:4]}" for s in seasons]

  # Precompute matchup lookup once
  matchup_cols_to_use = [
    'GAME_ID', 'TEAM_ABBREVIATION',
    'OPP_ALLOWED_PTS_L5_AVG', 'OPP_ALLOWED_PTS_CUM_AVG',
    'OPP_ALLOWED_REB_L5_AVG', 'OPP_ALLOWED_REB_CUM_AVG',
    'OPP_ALLOWED_AST_L5_AVG', 'OPP_ALLOWED_AST_CUM_AVG',
    'OPP_ALLOWED_3PM_L5_AVG', 'OPP_ALLOWED_3PM_CUM_AVG',
    'OPP_ALLOWED_STL_L5_AVG', 'OPP_ALLOWED_STL_CUM_AVG',
    'OPP_ALLOWED_BLK_L5_AVG', 'OPP_ALLOWED_BLK_CUM_AVG',
    'OPP_ALLOWED_TOV_L5_AVG', 'OPP_ALLOWED_TOV_CUM_AVG',
    'OPP_ALLOWED_FGM_L5_AVG', 'OPP_ALLOWED_FGM_CUM_AVG',
    'OPP_ALLOWED_FGA_L5_AVG', 'OPP_ALLOWED_FGA_CUM_AVG',
    'OPP_ALLOWED_3PA_L5_AVG', 'OPP_ALLOWED_3PA_CUM_AVG',
    'OPP_ALLOWED_FTM_L5_AVG', 'OPP_ALLOWED_FTM_CUM_AVG',
    'OPP_ALLOWED_FTA_L5_AVG', 'OPP_ALLOWED_FTA_CUM_AVG',
    'OPP_PACE_L5_AVG', 'OPP_PACE_CUM_AVG',
    'OPP_DEF_RATING_L5_AVG', 'OPP_DEF_RATING_CUM_AVG'
  ]
  matchup_lookup = matchup_df[matchup_cols_to_use].copy()

  matchup_opp_allowed = {
    'PTS': 'OPP_ALLOWED_PTS_L5_AVG',
    'REB': 'OPP_ALLOWED_REB_L5_AVG',
    'AST': 'OPP_ALLOWED_AST_L5_AVG',
    '3PM': 'OPP_ALLOWED_3PM_L5_AVG',
    'STL': 'OPP_ALLOWED_STL_L5_AVG',
    'BLK': 'OPP_ALLOWED_BLK_L5_AVG',
    'TOV': 'OPP_ALLOWED_TOV_L5_AVG',
    'FGM': 'OPP_ALLOWED_FGM_L5_AVG',
    'FGA': 'OPP_ALLOWED_FGA_L5_AVG',
    '3PA': 'OPP_ALLOWED_3PA_L5_AVG',
    'FTM': 'OPP_ALLOWED_FTM_L5_AVG',
    'FTA': 'OPP_ALLOWED_FTA_L5_AVG',
    'PACE': 'OPP_PACE_L5_AVG',
    'DEF_RATING': 'OPP_DEF_RATING_L5_AVG'
  }
  matchup_opp_source_cols = list(matchup_opp_allowed.values())

  COLS_FOR_PER = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'FGM', 'FTM', 'FGA', 'FTA']

  for player in players:
    time.sleep(8)
    print(" ")
    df_player = pd.DataFrame()
    player_id = player['id']
    player_name = player['full_name']

    # Get all raw data for all regular seasons and all playoffs and common player info
    try:
      df_inseason_full = pd.concat(PlayerGameLog(player_id=player_id, season=SeasonAll.all, season_type_all_star = "Regular Season").get_data_frames())
      df_playoffs_full = pd.concat(PlayerGameLog(player_id=player_id, season=SeasonAll.all, season_type_all_star = "Playoffs").get_data_frames())
      df_player_info =  pd.DataFrame(CommonPlayerInfo(player_id=player_id).get_data_frames()[0])
    except Exception as e:
      print("❌ Error on player:", player['full_name'])
      print(e)
      break;

    # Some Preprocessing of the data
    df_inseason_full = df_inseason_full[df_inseason_full["SEASON_ID"].isin(regular_season_ids)].copy()
    df_playoffs_full = df_playoffs_full[df_playoffs_full["SEASON_ID"].isin(playoff_season_ids)].copy()

    df_inseason_full['TEAM'] = df_inseason_full['MATCHUP'].str.split().str[0]
    df_inseason_full["HOME"] = df_inseason_full["MATCHUP"].str.contains("vs.").astype(int)
    df_inseason_full['MATCHUP'] = df_inseason_full['MATCHUP'].str.split().str[-1]
    df_inseason_full = df_inseason_full.rename(columns={'FG3M': '3PM', 'FG3A': '3PA', 'Player_ID': 'PLAYER_ID'})
    df_inseason_full["POSTSEASON"] = 0

    df_playoffs_full['TEAM'] = df_playoffs_full['MATCHUP'].str.split().str[0]
    df_playoffs_full['HOME'] = df_playoffs_full['MATCHUP'].str.contains("vs.").astype(int)
    df_playoffs_full['MATCHUP'] = df_playoffs_full['MATCHUP'].str.split().str[-1]
    df_playoffs_full = df_playoffs_full.rename(columns={'FG3M': '3PM', 'FG3A': '3PA', 'Player_ID': 'PLAYER_ID'})
    df_playoffs_full["POSTSEASON"] = 1

    for i in range(len(regular_season_ids)):
      df_inseason = df_inseason_full[df_inseason_full["SEASON_ID"] == regular_season_ids[i]]
      df_playoffs = df_playoffs_full[df_playoffs_full["SEASON_ID"] == playoff_season_ids[i]]

      try:
        df_year = pd.concat([x for x in [df_inseason, df_playoffs] if not x.empty], ignore_index=True)
      except ValueError:
        print(f"Skipped {player_name} {seasons[i]} (no games)")
        continue

      df_year = df_year.rename(columns={"Game_ID": "GAME_ID"})

      df_year = df_year.merge(matchup_lookup, left_on=['GAME_ID', 'TEAM'],right_on=['GAME_ID', 'TEAM_ABBREVIATION'],how='left')
      df_year = df_year.drop(columns=['TEAM_ABBREVIATION'])

      # Calculating raw data features for the season
      df_year['GAME_DATE'] = pd.to_datetime(df_year['GAME_DATE'], format='mixed', errors='raise')
      df_year = df_year.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)

      df_year['GAME_DIFF'] = df_year['GAME_DATE'].diff().dt.days
      df_year['BACK_TO_BACK'] = (df_year["GAME_DIFF"] == 1).astype(int)
      df_year['PLAYER_REST_DAYS'] = df_year['GAME_DIFF'].fillna(3).clip(0, 5)

      df_year['PRA'] = df_year['PTS'] + df_year['REB'] + df_year['AST']
      df_year['PA'] = df_year['PTS'] + df_year['AST']
      df_year['PR'] = df_year['PTS'] + df_year['REB']
      df_year['RA'] = df_year['REB'] + df_year['AST']
      df_year['SB'] = df_year['STL'] + df_year['BLK']

      ts_denom = 2 * (df_year['FGA'] + 0.44 * df_year['FTA']).replace(0, np.nan)
      df_year['TS%'] = 100 * df_year['PTS'] / ts_denom

      df_year["USG"] = ((df_year["FGA"] + 0.44 * df_year["FTA"] + df_year["TOV"])/ df_year["MIN"].replace(0, np.nan))

      df_year['PTS_PRODUCED'] = df_year['PTS'] + (df_year['AST'] * 2)
      df_year['PLAYER_POSSESSIONS'] = df_year['FGA'] + 0.44 * df_year['FTA'] + df_year['TOV']
      df_year['OFF_RATING'] = 100 * (df_year['PTS_PRODUCED'] / df_year['PLAYER_POSSESSIONS'].replace(0, np.nan))

      df_year['POSITION'] = df_player_info['POSITION'].iloc[0]
      df_year['HEIGHT'] = (int(df_player_info['HEIGHT'].iloc[0].split('-')[0]) * 12 + int(df_player_info['HEIGHT'].iloc[0].split('-')[1]))
      df_year['WEIGHT'] = pd.to_numeric(df_player_info['WEIGHT'].iloc[0])
      df_year['PLAYER_NAME'] = player_name
      df_year['SEASON_YEAR'] = seasons[i]

      df_year['FATIGUE_FACTOR'] = df_year['OPP_PACE_L5_AVG'] * df_year['PLAYER_REST_DAYS']

      df_year['AST_VULN_RATIO'] = (df_year['OPP_ALLOWED_AST_L5_AVG'] /df_year['OPP_ALLOWED_AST_CUM_AVG'].replace(0, np.nan)).fillna(1.0)
      df_year['REB_VULN_RATIO'] = (df_year['OPP_ALLOWED_REB_L5_AVG'] /df_year['OPP_ALLOWED_REB_CUM_AVG'].replace(0, np.nan)).fillna(1.0)

      print("Adding", player_name, seasons[i])
      print(f"  Season {seasons[i]}: {len(df_year)} games")

      df_player = pd.concat([df_player, df_year], ignore_index=True)

    if df_player.empty:
      print(f"Skipped {player_name} (no usable games)")
      continue

    print(f"\n📊 After all seasons: {len(df_player)} rows")

    # Adding on to previous history requires grouping by player name for matchup specific stats
    # Matchup specific stats may depend on previous seasons so we concatenate with existing df
    # Matchup context history (dedupe if df already has exploded rows)
    df_player = df_player.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)

    matchup_cols = list(dict.fromkeys([
      'PLAYER_ID', 'GAME_DATE', 'GAME_ID', 'MATCHUP',
      *ADVANCED_COLS,
      *COLS_FOR_PER,
      'OFF_RATING',
      'OPP_ALLOWED_PTS_L5_AVG', 'OPP_ALLOWED_PTS_CUM_AVG',
      'OPP_ALLOWED_REB_L5_AVG', 'OPP_ALLOWED_REB_CUM_AVG',
      'OPP_ALLOWED_AST_L5_AVG', 'OPP_ALLOWED_AST_CUM_AVG',
      'OPP_ALLOWED_STL_L5_AVG', 'OPP_ALLOWED_STL_CUM_AVG',
      'OPP_ALLOWED_BLK_L5_AVG', 'OPP_ALLOWED_BLK_CUM_AVG',
      'OPP_ALLOWED_TOV_L5_AVG', 'OPP_ALLOWED_TOV_CUM_AVG',
      'OPP_ALLOWED_3PM_L5_AVG', 'OPP_ALLOWED_3PM_CUM_AVG',
      'OPP_ALLOWED_FGM_L5_AVG', 'OPP_ALLOWED_FGM_CUM_AVG',
      'OPP_ALLOWED_FGA_L5_AVG', 'OPP_ALLOWED_FGA_CUM_AVG',
      'OPP_ALLOWED_3PA_L5_AVG', 'OPP_ALLOWED_3PA_CUM_AVG',
      'OPP_ALLOWED_FTM_L5_AVG', 'OPP_ALLOWED_FTM_CUM_AVG',
      'OPP_ALLOWED_FTA_L5_AVG', 'OPP_ALLOWED_FTA_CUM_AVG',
      'OPP_PACE_L5_AVG', 'OPP_PACE_CUM_AVG',
      'OPP_DEF_RATING_L5_AVG', 'OPP_DEF_RATING_CUM_AVG'
    ]))

    if df is not None and not df.empty:
      df_player_history = (df[df['PLAYER_ID'] == player_id].sort_values('GAME_DATE').drop_duplicates(subset=['PLAYER_ID', 'GAME_ID']))
      df_matchup_context = pd.concat([df_player_history[matchup_cols], df_player[matchup_cols]], ignore_index=True)
      df_matchup_context = df_matchup_context.sort_values('GAME_DATE').reset_index(drop=True)
      df_matchup_context['_is_new'] = False
      df_matchup_context.loc[df_matchup_context.index[-len(df_player):], '_is_new'] = True
    else:
      df_matchup_context = df_player.sort_values('GAME_DATE').reset_index(drop=True)
      df_matchup_context['_is_new'] = True

    # Calculating matchup based stats w/ last 4 games against that matchup
    # Vectorized caching to improve performance
    matchup_keys = [df_matchup_context['PLAYER_ID'], df_matchup_context['MATCHUP']]
    grp_matchup = df_matchup_context.groupby(['PLAYER_ID', 'MATCHUP'], sort=False)

    last_matchup = grp_matchup[ADVANCED_COLS].shift(1)
    last_matchup.columns = [f'LAST_MATCHUP_{c}' for c in ADVANCED_COLS]

    # Last-4 matchup averages/stds
    last_matchup_avg = _grouped_rolling_from_shifted(shifted=last_matchup,keys=matchup_keys,window=4,min_periods=1,agg="mean")
    last_matchup_avg.columns = [f'MATCHUP_L4_AVG_{c}' for c in ADVANCED_COLS]

    last_matchup_std = _grouped_rolling_from_shifted(shifted=last_matchup,keys=matchup_keys,window=4,min_periods=2,agg="std",ddof=0).clip(lower=1e-6)
    last_matchup_std.columns = [f'MATCHUP_L4_STD_{c}' for c in ADVANCED_COLS]

    # Opponent-allowed matchup block
    shifted_opp_allowed = grp_matchup[matchup_opp_source_cols].shift(1)
    matchup_opp_allowed_l4 = _grouped_rolling_from_shifted(shifted=shifted_opp_allowed,keys=matchup_keys,window=4,min_periods=1,agg="mean")
    matchup_opp_allowed_l4.columns = [f"MATCHUP_OPP_ALLOWED_{k}_L4" for k in matchup_opp_allowed.keys()]

    # Last-4 sums for custom grouped PER
    shifted_per_source = grp_matchup[COLS_FOR_PER].shift(1)
    cum_vs_team = _grouped_rolling_from_shifted(shifted=shifted_per_source,keys=matchup_keys,window=4,min_periods=1,agg="sum")
    cum_vs_team.columns = [f'CUM_AVG_VS_TEAM_{c}' for c in COLS_FOR_PER]

    df_matchup_context['GP_AGAINST_TEAM'] = grp_matchup.cumcount().clip(upper=4)

    shifted_pm = grp_matchup['PLUS_MINUS'].shift(1)
    df_matchup_context['PLUS_MINUS_GROUPED'] = _grouped_rolling_from_shifted(shifted=shifted_pm,keys=matchup_keys,window=4,min_periods=1,agg="mean")

    shifted_off = grp_matchup['OFF_RATING'].shift(1)
    df_matchup_context['OFF_RATING_GROUPED'] = _grouped_rolling_from_shifted(shifted=shifted_off,keys=matchup_keys,window=4,min_periods=1,agg="mean")

    df_matchup_context['PER_GROUPED'] = (
      (cum_vs_team['CUM_AVG_VS_TEAM_PTS'] + cum_vs_team['CUM_AVG_VS_TEAM_REB'] + cum_vs_team['CUM_AVG_VS_TEAM_AST'] +
       cum_vs_team['CUM_AVG_VS_TEAM_STL'] + cum_vs_team['CUM_AVG_VS_TEAM_BLK']) -
      ((cum_vs_team['CUM_AVG_VS_TEAM_FGA'] - cum_vs_team['CUM_AVG_VS_TEAM_FGM']) +
       (cum_vs_team['CUM_AVG_VS_TEAM_FTA'] - cum_vs_team['CUM_AVG_VS_TEAM_FTM']) +
       cum_vs_team['CUM_AVG_VS_TEAM_TOV'])
    ) / df_matchup_context['GP_AGAINST_TEAM'].replace(0, np.nan)

    df_matchup_context = pd.concat(
      [
        df_matchup_context,
        last_matchup,
        last_matchup_avg,
        last_matchup_std,
        cum_vs_team,
        matchup_opp_allowed_l4
      ],
      axis=1
    )

    MATCHUP_FEATURE_COLS = (
      [f'LAST_MATCHUP_{col}' for col in ADVANCED_COLS] +
      [f'MATCHUP_L4_AVG_{col}' for col in ADVANCED_COLS] +
      [f'MATCHUP_L4_STD_{col}' for col in ADVANCED_COLS] +
      ['PER_GROUPED', 'PLUS_MINUS_GROUPED', 'OFF_RATING_GROUPED', 'GP_AGAINST_TEAM'] +
      [f"MATCHUP_OPP_ALLOWED_{metric}_L4" for metric in MATCHUP_ALLOWED_METRICS_W_PACE_DEF]
    )

    df_matchup_new = df_matchup_context.loc[df_matchup_context['_is_new'], MATCHUP_FEATURE_COLS].reset_index(drop=True)
    df_player = pd.concat([df_player.reset_index(drop=True), df_matchup_new], axis=1)

    # Calculating cumulative averages, last 5 avgs and stds for all stats up to but not including the current game grouped by season
    # Using vectorized caching to improve performance
    df_player = df_player.sort_values(['SEASON_YEAR', 'GAME_DATE', 'GAME_ID']).reset_index(drop=True)
    season_keys = [df_player['SEASON_YEAR']]
    grp_season = df_player.groupby('SEASON_YEAR', sort=False)

    shifted_adv = grp_season[ADVANCED_COLS].shift(1)

    expanding_mean = _grouped_expanding_from_shifted(shifted=shifted_adv,keys=season_keys,agg="mean")
    expanding_mean.columns = [f"CUM_AVG_{c}" for c in ADVANCED_COLS]

    expanding_std = _grouped_expanding_from_shifted(shifted=shifted_adv,keys=season_keys,agg="std",ddof=0)
    expanding_std.columns = [f"STD_CUM_AVG_{c}" for c in ADVANCED_COLS]

    last5_mean = _grouped_rolling_from_shifted(shifted=shifted_adv,keys=season_keys,window=5,min_periods=1,agg="mean")
    last5_mean.columns = [f"L5_AVG_{c}" for c in ADVANCED_COLS]

    last5_std = _grouped_rolling_from_shifted(shifted=shifted_adv,keys=season_keys,window=5,min_periods=1,agg="std",ddof=0).clip(lower=1e-6)
    last5_std.columns = [f"STD_L5_AVG_{c}" for c in ADVANCED_COLS]

    df_player = pd.concat([df_player, expanding_mean, expanding_std, last5_mean, last5_std], axis=1)

    # Compute previously leaked features using lagged proxies
    pre_min = df_player['L5_AVG_MIN'].combine_first(df_player['CUM_AVG_MIN'])
    pre_usg = df_player['L5_AVG_USG'].combine_first(df_player['CUM_AVG_USG'])

    df_player['PACE_IMPACT_POSS'] = ((pre_min / 48.0) *(df_player['OPP_PACE_L5_AVG'] - df_player['OPP_PACE_CUM_AVG']).fillna(0))
    df_player['DEF_VS_VOL'] = df_player['OPP_DEF_RATING_L5_AVG'] * pre_usg

    df_player = df_player[stat_utils.columns_to_keep_first_filter]
    print(f"After first filter: {len(df_player)} rows")

    df_player = df_player.sort_values(["GAME_DATE", "GAME_ID"]).reset_index(drop=True)

    # Getting Last 10 games for every row as a matrix
    last10_mats = {}
    last5_mats = {}

    for col in STAT_COLS:
      values = df_player[col].to_numpy()
      m10 = build_last10_matrix(values)          # shape (N, 10)
      m5 = m10[:, -5:]                           # last 5 of last10
      last10_mats[col] = m10
      last5_mats[col] = m5

    # Random parlay line generation for all stats each row will get an array of 28 parlays
    print(f"\n🔍 DEBUG: Checking parlay input columns for {player_name}")

    for col in STAT_COLS:
      avg = df_player[f'L5_AVG_{col}'].to_numpy()
      cumm_avg = df_player[f'CUM_AVG_{col}'].to_numpy()
      matchup_avg = df_player[f'MATCHUP_L4_AVG_{col}'].to_numpy()
      gp_against = df_player['GP_AGAINST_TEAM'].to_numpy()

      try:
        parlays, anchor = generate_parlays_batch(
          avg=avg,
          cumm_avg=cumm_avg,
          matchup_avg=matchup_avg,
          std_last5=df_player[f'STD_L5_AVG_{col}'].to_numpy(),
          std_cum=df_player[f'STD_CUM_AVG_{col}'].to_numpy(),
          last10_mat=last10_mats[col],
          gp_against_team=gp_against
        )

        df_player[f'{col}_ANCHOR'] = anchor

        parlays = sanitize_parlay_matrix(parlays, MAX_LINES)

        num_valid = np.isfinite(parlays).any(axis=1).sum()
        print(f"  {col}: {num_valid}/{len(parlays)} rows have valid parlays")

        df_player[f'PL_{col}'] = list(parlays)

      except Exception as e:
        print(f"Error with player {player['full_name']} and column {col}: {e}")
        raise RuntimeError(f"Parlay failure: {player_name}, {col}") from e

    # Calculating more important data features
    momentum_cols = {}
    for col in ADVANCED_COLS:
      momentum_cols[f'{col}_MOMENTUM'] = df_player[f'L5_AVG_{col}'] - df_player[f'CUM_AVG_{col}']

    df_player = pd.concat([df_player, pd.DataFrame(momentum_cols),], axis=1)

    cum_avg_per_min_cols = {}
    last5_avg_per_min_cols = {}
    momentum_x_vol_col = {}
    games_last3_days_cols = {}
    games_last7_days_cols = {}

    for col in STAT_COLS:
      cum_avg_per_min_cols[f'CUM_AVG_{col}_PER_MIN'] = df_player[f'CUM_AVG_{col}'] / df_player['CUM_AVG_MIN'].replace(0, np.nan)
      last5_avg_per_min_cols[f'L5_{col}_PER_MIN'] = df_player[f'L5_AVG_{col}'] / df_player['L5_AVG_MIN'].replace(0, np.nan)
      momentum_x_vol_col[f'{col}_MOMENTUM_X_VOL'] = df_player[f'{col}_MOMENTUM'] * df_player[f'STD_L5_AVG_{col}']

    grp_season_now = df_player.groupby("SEASON_YEAR", sort=False)
    games_last3_days_cols["GAMES_L3_DAYS"] = grp_season_now["GAME_DATE"].transform(lambda s: count_games_last_days(s, 3))
    games_last7_days_cols["GAMES_L7_DAYS"] = grp_season_now["GAME_DATE"].transform(lambda s: count_games_last_days(s, 7))

    df_player = pd.concat([
        df_player,
        pd.DataFrame(cum_avg_per_min_cols),
        pd.DataFrame(last5_avg_per_min_cols),
        pd.DataFrame(momentum_x_vol_col),
        pd.DataFrame(games_last3_days_cols),
        pd.DataFrame(games_last7_days_cols),
    ], axis=1)

    # Exploding parlay lines increases size of dataframe x35
    N = len(df_player)
    line_idx = np.tile(np.arange(MAX_LINES), N)
    row_idx = np.repeat(np.arange(N), MAX_LINES)

    df_long = df_player.iloc[row_idx].copy()
    df_long["LINE_IDX"] = line_idx

    for col in STAT_COLS:
      df_long[f'PL_{col}'] = np.concatenate(df_player[f'PL_{col}'].values)
      df_long[f'{col}_DIST_FROM_ANCHOR'] = np.abs(df_long[f'PL_{col}'] - df_long[f'{col}_ANCHOR'])

    df_player = df_long
    print(f"After parlay explosion: {len(df_player)} rows (should be ~35x)")  # ← ADD THIS

    pl_cols = [f'PL_{c}' for c in STAT_COLS]

    # Keep only rows where at least one stat line exists
    keep_mask = ~df_player[pl_cols].isna().all(axis=1).to_numpy()

    # Filter df_player and ALSO filter the base mapping arrays
    df_player = df_player.loc[keep_mask].reset_index(drop=True)
    row_idx = row_idx[keep_mask]
    line_idx = line_idx[keep_mask]

    print(f"After dropping NaN parlays: {len(df_player)} rows")

    # print("\n🔍 Checking exploded parlay columns:")
    # for col in STAT_COLS[:3]:  # Check first 3
    #   non_null = df_player[f'PL_{col}'].notna().sum()
    #   print(f"  PL_{col}: {non_null}/{len(df_player)} non-null")
    #   # Show some sample values
    #   sample = df_player[f'PL_{col}'].dropna().head(3)
    #   print(f"    Samples: {sample.tolist()}")

    # pl_cols = [f'PL_{c}' for c in STAT_COLS]
    # df_player = df_player.dropna(subset=pl_cols, how='all').reset_index(drop=True)
    # df_player = df_player.reset_index(drop=True)

    # print(f"After dropping NaN parlays: {len(df_player)} rows")

    # OPTION A overwrite: OVER_PL_RATE_* computed from last-N PRIOR games across seasons
    for col in STAT_COLS:
      base_idx = row_idx  # maps exploded row -> base game row (pre-explosion)
      line_vals = df_player[f"PL_{col}"].to_numpy(dtype=np.float64)

      m10 = last10_mats[col][base_idx]  # (N*MAX_LINES, 10)
      m5  = last5_mats[col][base_idx]   # (N*MAX_LINES, 5)

      valid10 = np.isfinite(m10)
      valid5  = np.isfinite(m5)

      gt10 = (m10 > line_vals[:, None]) & valid10
      gt5  = (m5  > line_vals[:, None]) & valid5

      denom10 = np.maximum(valid10.sum(axis=1), 1)
      denom5  = np.maximum(valid5.sum(axis=1), 1)

      over10 = gt10.sum(axis=1) / denom10
      over5  = gt5.sum(axis=1)  / denom5

      # If no prior games (denom=1 but valid sum=0), you may want neutral 0.5 instead of 0.0.
      # Current: 0.0
      over10 = np.where(valid10.sum(axis=1) == 0, 0.0, over10)
      over5  = np.where(valid5.sum(axis=1) == 0, 0.0, over5)

      df_player[f"OVER_PL_RATE_{col}_L10"] = over10.astype(np.float32)
      df_player[f"OVER_PL_RATE_{col}_L5"]  = over5.astype(np.float32)

    # Parlay prediction based on the actual stats of the player
    for col in STAT_COLS:
      df_player[f'TARGET_{col}'] = np.where(df_player[f'PL_{col}'].notna(),(df_player[col] > df_player[f'PL_{col}']).astype(float),np.nan)

    # Calculating more Data features
    line_diff_cols = {}
    for col in STAT_COLS:
      line_diff_cols[f'{col}_LINE_DIFF'] = df_player[f'PL_{col}'] - df_player[f'L5_AVG_{col}']

    z_line_cols = {}
    z_recent_cols = {}
    z_matchup_cols = {}
    line_diff_x_min_cols = {}
    min_deviation_cols = {}

    for col in STAT_COLS:
      z_line_cols[f'{col}_Z_LINE'] = ((df_player[f'PL_{col}'] - df_player[f'CUM_AVG_{col}']) /(df_player[f'STD_L5_AVG_{col}'] + 1e-6)).clip(-6, 6)
      z_recent_cols[f'{col}_Z_RECENT'] = ((df_player[f'PL_{col}'] - df_player[f'L5_AVG_{col}']) /(df_player[f'STD_L5_AVG_{col}'] + 1e-6)).clip(-6, 6)
      z_matchup_cols[f'{col}_Z_MATCHUP'] = ((df_player[f'PL_{col}'] - df_player[f'MATCHUP_L4_AVG_{col}']) /(df_player[f'MATCHUP_L4_STD_{col}'] + 1e-6)).clip(-6, 6)
      line_diff_x_min_cols[f'{col}_LINE_DIFF_X_MIN'] =  line_diff_cols[f'{col}_LINE_DIFF'] * df_player['CUM_AVG_MIN']

    df_player = pd.concat([
        df_player,
        pd.DataFrame(z_line_cols),
        pd.DataFrame(z_recent_cols),
        pd.DataFrame(z_matchup_cols),
        pd.DataFrame(line_diff_x_min_cols),
        pd.DataFrame(line_diff_cols),
    ], axis=1)

    gp_weight = (df_player['GP_AGAINST_TEAM'] / 4).clip(0, 1)
    for col in STAT_COLS:
      df_player[f'{col}_Z_MATCHUP'] *= gp_weight

    df_player = df_player[stat_utils.columns_to_keep_second_filter]
    print(f"After second filter: {len(df_player)} rows")

    df = pd.concat([df, df_player], ignore_index=True)
    print(f"Total df size: {len(df)} rows\n")

  print("Done")
  return df

In [ ]:
years = ['2015-16','2016-17','2017-18','2018-19','2019-20','2020-21','2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
matchup_engine = build_matchup_engine(years)

Fetching Team Logs for 2015-16...
Fetching Team Logs for 2016-17...
Fetching Team Logs for 2017-18...
Fetching Team Logs for 2018-19...
Fetching Team Logs for 2019-20...
Fetching Team Logs for 2020-21...
Fetching Team Logs for 2021-22...
Fetching Team Logs for 2022-23...
Fetching Team Logs for 2023-24...
Fetching Team Logs for 2024-25...
Fetching Team Logs for 2025-26...


/tmp/ipykernel_5202/672668589.py:159: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_log = pd.concat([df_inseason, df_playoffs], ignore_index=True)



✅ Final matchup_engine shape: (27542, 30)
Final columns: ['GAME_ID', 'TEAM_ABBREVIATION', 'OPP_ALLOWED_PTS_L5_AVG', 'OPP_ALLOWED_PTS_CUM_AVG', 'OPP_ALLOWED_REB_L5_AVG', 'OPP_ALLOWED_REB_CUM_AVG', 'OPP_ALLOWED_AST_L5_AVG', 'OPP_ALLOWED_AST_CUM_AVG', 'OPP_ALLOWED_STL_L5_AVG', 'OPP_ALLOWED_STL_CUM_AVG', 'OPP_ALLOWED_BLK_L5_AVG', 'OPP_ALLOWED_BLK_CUM_AVG', 'OPP_ALLOWED_TOV_L5_AVG', 'OPP_ALLOWED_TOV_CUM_AVG', 'OPP_ALLOWED_3PM_L5_AVG', 'OPP_ALLOWED_3PM_CUM_AVG', 'OPP_ALLOWED_FGM_L5_AVG', 'OPP_ALLOWED_FGM_CUM_AVG', 'OPP_ALLOWED_FGA_L5_AVG', 'OPP_ALLOWED_FGA_CUM_AVG', 'OPP_ALLOWED_3PA_L5_AVG', 'OPP_ALLOWED_3PA_CUM_AVG', 'OPP_ALLOWED_FTM_L5_AVG', 'OPP_ALLOWED_FTM_CUM_AVG', 'OPP_ALLOWED_FTA_L5_AVG', 'OPP_ALLOWED_FTA_CUM_AVG', 'OPP_PACE_L5_AVG', 'OPP_PACE_CUM_AVG', 'OPP_DEF_RATING_L5_AVG', 'OPP_DEF_RATING_CUM_AVG']


In [ ]:
from google.colab import files

all_players = players.get_players()
active_players = [player for player in all_players if player['is_active']]

required_names = stat_utils.required_names
active_names = {p['full_name'] for p in active_players}
missing = required_names - active_names
if missing:
  raise ValueError(f"Missing players in active_players: {sorted(missing)}")

test_players = [p for p in active_players if p['full_name'] in required_names]
df = pd.DataFrame()


try:
    df = create_df(df, test_players, years, matchup_engine)
except Exception as e:
    print("⚠️ create_df failed, continuing pipeline")
    print(e)

df.to_parquet('df_XGB.parquet', index=False)

df.sample(n=10, random_state=42)

!zip -q  df_XGB.zip df_XGB.parquet
files.download('df_XGB.zip')


Streaming output truncated to the last 5000 lines.
Skipped Dyson Daniels 2016-17 (no games)
Skipped Dyson Daniels 2017-18 (no games)
Skipped Dyson Daniels 2018-19 (no games)
Skipped Dyson Daniels 2019-20 (no games)
Skipped Dyson Daniels 2020-21 (no games)
Skipped Dyson Daniels 2021-22 (no games)
Adding Dyson Daniels 2022-23
  Season 2022-23: 59 games
Adding Dyson Daniels 2023-24
  Season 2023-24: 64 games
Adding Dyson Daniels 2024-25
  Season 2024-25: 76 games
Adding Dyson Daniels 2025-26
  Season 2025-26: 61 games

📊 After all seasons: 260 rows
After first filter: 260 rows

🔍 DEBUG: Checking parlay input columns for Dyson Daniels
  PTS: 256/260 rows have valid parlays
  REB: 256/260 rows have valid parlays
  AST: 256/260 rows have valid parlays
  STL: 256/260 rows have valid parlays
  BLK: 256/260 rows have valid parlays
  PRA: 256/260 rows have valid parlays
  PA: 256/260 rows have valid parlays
  PR: 256/260 rows have valid parlays
  RA: 256/260 rows have valid parlays
  SB: 256/260

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, Input, Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, log_loss, brier_score_loss
from tensorflow.keras.callbacks import EarlyStopping
import pickle
import shutil
from google.colab import files
import os

In [ ]:
CAT_COLS = ["PLAYER_NAME", "TEAM", "POSITION", "MATCHUP"]

def build_union_category_mappings(parquet_paths, cat_cols=CAT_COLS):
  # Collect all unique values across all datasets for each column
  union_values = {c: set() for c in cat_cols}

  for path in parquet_paths:
    df = pd.read_parquet(path, columns=cat_cols)
    for c in cat_cols:
      union_values[c].update(df[c].dropna().astype(str).unique().tolist())

  mappings = {}
  for c in cat_cols:
    ordered = sorted(union_values[c])
    mappings[c] = {val: i for i, val in enumerate(ordered)}
    mappings[c]["UNK"] = -1

  return mappings

category_mappings = build_union_category_mappings(parquet_paths=["df_XGB.parquet"])
with open("category_mappings.json", "w") as f:
    json.dump(category_mappings, f, indent=2)

print("Saved canonical category_mappings.json")

!zip -q  category_mappings.zip category_mappings.json
files.download('category_mappings.zip')



Saved canonical category_mappings.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
CAT_COLS = ["PLAYER_NAME", "TEAM", "POSITION", "MATCHUP"]

def apply_category_mappings(df, category_mappings, cat_cols=CAT_COLS):
    """
    Apply category mappings without DataFrame fragmentation
    Creates all new columns at once using pd.concat
    """
    # Create dictionary to store all new columns
    new_columns = {}

    for c in cat_cols:
        if c in df.columns:
            new_columns[f"{c}_ID"] = (
                df[c]
                .astype(str)
                .map(category_mappings[c])
                .fillna(-1)
                .astype(int)
            )

    # Add all new columns at once (no fragmentation!)
    if new_columns:
        df = pd.concat([df, pd.DataFrame(new_columns, index=df.index)], axis=1)

    return df

In [ ]:
# =========================
# STAGE 2: TUNING + CALIBRATION SELECTION (COLAB) - FIXED
#  - GPU optimized
#  - only 2 calibration variants + raw
#  - smaller HP grid (3 configs)
#  - checkpoint saves after each stat
#  - never silently "loses" a stat: logs SKIPPED / FAILED rows
# =========================

import os
import gc
import json
import time
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score
from google.colab import files

import stat_utils

# -------------------------
# QUIET WARNINGS
# -------------------------
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.calibration")
warnings.filterwarnings("ignore", category=RuntimeWarning)

# -------------------------
# CONFIG
# -------------------------
SEASON_HOLDOUT = "2025-26"
CAL_FRAC = 0.70

STATS_TO_RUN = list(stat_utils.STAT_COLS)  # all 17
print("Running stats:", STATS_TO_RUN)

# Tight-line definition
TIGHT_ALPHA = 0.25
TIGHT_FLOOR_DEFAULT = 1.0
TIGHT_FLOOR_BY_STAT = {
  "PTS": 1.5, "PRA": 1.5, "PA": 1.5, "PR": 1.5, "RA": 1.5
}

CONF_THRESHOLDS = [0.55, 0.60, 0.65, 0.70]

N_PLAYERS_FAST = 45  # set None for all players

# Smaller HP grid (cfg0, cfg2, cfg4)
HP_GRID = [
  {"max_depth": 3, "min_child_weight": 2, "subsample": 0.7, "colsample_bytree": 0.5, "reg_alpha": 0.0, "gamma": 0.0},  # shallow
  {"max_depth": 4, "min_child_weight": 3, "subsample": 0.7, "colsample_bytree": 0.6, "reg_alpha": 0.1, "gamma": 0.1},  # balanced
  {"max_depth": 5, "min_child_weight": 4, "subsample": 0.8, "colsample_bytree": 0.6, "reg_alpha": 0.0, "gamma": 0.1},  # deeper
]

# GPU-optimized base params (Stage 2 speed)
BASE_XGB_PARAMS = {
  "objective": "binary:logistic",
  "eval_metric": "logloss",
  "n_estimators": 1200,
  "learning_rate": 0.02,
  "reg_lambda": 1.0,
  "early_stopping_rounds": 50,
  "n_jobs": -1,
  "random_state": 42,
  "tree_method": "hist"
}

# Minimum thresholds (we still LOG when we skip)
MIN_TRAIN = 1000
MIN_CAL = 250
MIN_BT = 250
MIN_CAL_TIGHT = 200

# Output paths (checkpoint)
OUT_RESULTS = "stage2_tuning_results.csv"
OUT_BUCKETS = "stage2_bucket_sanity.csv"

# -------------------------
# MEMORY OPTIMIZATION
#  - force TARGET_* to float32 so means don't overflow in float16
# -------------------------
def optimize_dtypes(df):
  print(f"Before optimization: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

  for col in df.columns:
    if col.startswith("TARGET_"):
      df[col] = df[col].astype(np.float32, copy=False)
      continue

    col_type = df[col].dtype

    if col_type == "object" or col_type.name == "category":
      continue

    if col_type == "int64":
      c_min = df[col].min()
      c_max = df[col].max()
      if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
        df[col] = df[col].astype(np.int8)
      elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
        df[col] = df[col].astype(np.int16)
      elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
        df[col] = df[col].astype(np.int32)

    elif col_type == "float64":
      c_min = df[col].min()
      c_max = df[col].max()
      if pd.notna(c_min) and pd.notna(c_max):
        if abs(c_min) < 65000 and abs(c_max) < 65000:
          test_val = df[col].iloc[0] if len(df[col]) > 0 else 0
          if pd.notna(test_val):
            precision_loss = abs(test_val - np.float16(test_val)) / (abs(test_val) + 1e-10)
            if precision_loss < 0.01:
              df[col] = df[col].astype(np.float16)
              continue
      df[col] = df[col].astype(np.float32)

  print(f"After optimization: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
  print(f"  Float16 columns: {len(df.select_dtypes(include=['float16']).columns)}")
  print(f"  Float32 columns: {len(df.select_dtypes(include=['float32']).columns)}")
  return df

# -------------------------
# SPLITS (game-level, chronological)
# -------------------------
def build_game_keys(df):
  return list(zip(df["PLAYER_ID"].astype(int).values, df["GAME_ID"].astype(str).values))

def make_splits(df, season_holdout=SEASON_HOLDOUT, cal_frac=CAL_FRAC):
  train_mask = df["SEASON_YEAR"] != season_holdout
  df_train = df[train_mask]
  df_hold = df[~train_mask].copy()

  if len(df_hold) == 0:
    raise ValueError(f"No rows found for holdout season {season_holdout}")

  g = df_hold.groupby(["PLAYER_ID", "GAME_ID"], sort=False)["GAME_DATE"].min().reset_index()
  g = g.sort_values(["GAME_DATE", "PLAYER_ID", "GAME_ID"]).reset_index(drop=True)

  n_games = len(g)
  cut = int(np.floor(cal_frac * n_games))
  cut = max(1, min(cut, n_games - 1))

  cal_games = set(zip(g.iloc[:cut]["PLAYER_ID"].astype(int), g.iloc[:cut]["GAME_ID"].astype(str)))
  bt_games  = set(zip(g.iloc[cut:]["PLAYER_ID"].astype(int), g.iloc[cut:]["GAME_ID"].astype(str)))

  hold_keys = build_game_keys(df_hold)
  hold_in_cal = np.array([k in cal_games for k in hold_keys], dtype=bool)
  hold_in_bt  = np.array([k in bt_games  for k in hold_keys], dtype=bool)

  df_cal = df_hold.iloc[np.where(hold_in_cal)[0]]
  df_bt  = df_hold.iloc[np.where(hold_in_bt)[0]]
  return df_train, df_cal, df_bt

def tight_mask_for_stat(df, stat, alpha=TIGHT_ALPHA):
  dist = df[f"{stat}_DIST_FROM_ANCHOR"].to_numpy()
  std5 = df[f"STD_L5_AVG_{stat}"].to_numpy()
  stdc = df[f"STD_CUM_AVG_{stat}"].to_numpy()
  std = np.where(np.isfinite(std5), std5, stdc)

  floor = TIGHT_FLOOR_BY_STAT.get(stat, TIGHT_FLOOR_DEFAULT)
  thresh = np.maximum(alpha * std, floor)
  return np.isfinite(dist) & np.isfinite(thresh) & (dist <= thresh)

# -------------------------
# METRICS (float64 to avoid overflow)
# -------------------------
def bucket_report_full(probs, y_true):
  probs = np.asarray(probs, dtype=np.float64)
  y_true = np.asarray(y_true, dtype=np.float64)

  m = np.isfinite(probs) & np.isfinite(y_true)
  probs = probs[m]
  y_true = y_true[m]
  if probs.size == 0:
    return {}

  conf = np.maximum(probs, 1.0 - probs)
  pred = (probs >= 0.5).astype(int)
  correct = (pred == y_true).astype(int)

  buckets = [
    (0.50, 0.55),
    (0.55, 0.60),
    (0.60, 0.65),
    (0.65, 0.70),
    (0.70, 0.80),
    (0.80, 0.90),
    (0.90, 1.00),
  ]

  out = {}
  for lo, hi in buckets:
    key = f"{lo:.2f}-{hi:.2f}"
    in_bin = (conf >= lo) & (conf < hi) if hi < 1.0 else (conf >= lo) & (conf <= hi)

    out[f"{key}_n"] = int(in_bin.sum())
    if in_bin.sum() == 0:
      out[f"{key}_acc"] = np.nan
      out[f"{key}_avg_conf"] = np.nan
    else:
      out[f"{key}_acc"] = float(correct[in_bin].mean())
      out[f"{key}_avg_conf"] = float(conf[in_bin].mean())

  out["p99_conf"] = float(np.quantile(conf, 0.99))
  out["max_conf"] = float(conf.max())
  return out


def expected_calibration_error(probs, y_true, n_bins=10):
  probs = np.asarray(probs, dtype=np.float64)
  y_true = np.asarray(y_true, dtype=np.float64)
  mask = np.isfinite(probs) & np.isfinite(y_true)
  probs = probs[mask]
  y_true = y_true[mask]
  if probs.size == 0:
    return np.nan

  bins = np.linspace(0.0, 1.0, n_bins + 1)
  ece = 0.0
  for i in range(n_bins):
    lo, hi = bins[i], bins[i + 1]
    in_bin = (probs >= lo) & (probs < hi) if i < n_bins - 1 else (probs >= lo) & (probs <= hi)
    if not np.any(in_bin):
      continue
    ece += in_bin.mean() * abs(y_true[in_bin].mean() - probs[in_bin].mean())
  return float(ece)

def compute_metrics(probs, y_true, thresholds=CONF_THRESHOLDS):
  probs = np.asarray(probs, dtype=np.float64)
  y_true = np.asarray(y_true, dtype=np.float64)

  mask = np.isfinite(probs) & np.isfinite(y_true)
  probs = probs[mask]
  y_true = y_true[mask]
  if probs.size == 0:
    return None

  p_clip = np.clip(probs, 1e-7, 1 - 1e-7)

  out = {}
  out["logloss"] = float(log_loss(y_true, p_clip))
  out["brier"] = float(brier_score_loss(y_true, probs))
  try:
    out["auc"] = float(roc_auc_score(y_true, probs))
  except Exception:
    out["auc"] = np.nan

  out["base_acc"] = float(accuracy_score(y_true, (probs >= 0.5).astype(int)))
  out["ece10"] = expected_calibration_error(probs, y_true, n_bins=10)

  conf = np.maximum(probs, 1.0 - probs)
  out["p99_conf"] = float(np.quantile(conf, 0.99))
  out["max_conf"] = float(conf.max())

  for t in thresholds:
    m = (probs >= t) | (probs <= (1 - t))
    out[f"cov@{t:.2f}"] = float(m.mean())
    if m.sum() == 0:
      out[f"acc@{t:.2f}"] = np.nan
    else:
      out[f"acc@{t:.2f}"] = float(accuracy_score(y_true[m], (probs[m] >= 0.5).astype(int)))

  return out

# -------------------------
# FEATURES
# -------------------------
DROP_BASE_COLS = [
  "PLAYER_NAME", "PLAYER_ID", "TEAM", "MATCHUP", "POSITION",
  "SEASON_YEAR", "SEASON_ID", "GAME_DATE", "GAME_ID",
  # never include current-game outcomes
  "PTS", "REB", "AST", "STL", "BLK", "PRA", "PA", "PR", "RA", "SB",
  "TOV", "FTM", "FGM", "3PM", "FGA", "3PA", "FTA",
  "MIN", "PLUS_MINUS", "TS%", "USG", "OFF_RATING"
]

def build_feature_frame(df, target_stat, stat_cols, all_targets, feature_mode="full"):
  cols_to_drop = list(DROP_BASE_COLS) + list(all_targets)

  # Drop other stats' line-derived features
  for s in stat_cols:
    if s != target_stat:
      cols_to_drop.extend([
        f"PL_{s}",
        f"OVER_PL_RATE_{s}_L10",
        f"OVER_PL_RATE_{s}_L5",
        f"{s}_Z_LINE",
        f"{s}_Z_RECENT",
        f"{s}_Z_MATCHUP",
        f"{s}_LINE_DIFF_X_MIN",
        f"{s}_LINE_DIFF",
        f"{s}_DIST_FROM_ANCHOR",
        f"{s}_ANCHOR"
      ])

  X = df.drop(columns=cols_to_drop, errors="ignore")
  if feature_mode == "full":
    return X

  # reduced: drop most non-target-stat history columns by prefix
  drop_more = []
  for s in stat_cols:
    if s == target_stat:
      continue

    prefixes = [
      f"CUM_AVG_{s}", f"L5_AVG_{s}", f"STD_CUM_AVG_{s}", f"STD_L5_AVG_{s}",
      f"{s}_",
      f"LAST_MATCHUP_{s}", f"MATCHUP_L4_AVG_{s}", f"MATCHUP_L4_STD_{s}",
      f"CUM_AVG_{s}_PER_MIN", f"L5_{s}_PER_MIN"
    ]
    for p in prefixes:
      drop_more.extend([c for c in X.columns if c.startswith(p)])

    drop_more.extend([c for c in X.columns if c.startswith(f"OPP_ALLOWED_{s}_")])
    drop_more.extend([c for c in X.columns if c.startswith(f"MATCHUP_OPP_ALLOWED_{s}_")])

  if drop_more:
    drop_more = list(dict.fromkeys(drop_more))
    X = X.drop(columns=drop_more, errors="ignore")

  return X

# -------------------------
# CHECKPOINT WRITER
# -------------------------
def append_rows_to_csv(path, rows, header_if_new=True):
  if not rows:
    return
  df_out = pd.DataFrame(rows)
  write_header = header_if_new and (not os.path.exists(path))
  df_out.to_csv(path, mode="a", index=False, header=write_header)

# -------------------------
# MAIN
# -------------------------
print("Loading data...")
with open("category_mappings.json", "r") as f:
  category_mappings = json.load(f)

df = pd.read_parquet("df_XGB.parquet").replace([np.inf, -np.inf], np.nan)
df = optimize_dtypes(df)

# apply_category_mappings must already exist in your notebook
df = apply_category_mappings(df, category_mappings)

df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], errors="coerce")

# -------------------------
# OPTIONAL: SUBSAMPLE PLAYERS FOR SPEED (keeps overlap between train and holdout)
# -------------------------
if N_PLAYERS_FAST is not None:
  p_train = set(df.loc[df["SEASON_YEAR"] != SEASON_HOLDOUT, "PLAYER_ID"].dropna().astype(int).unique().tolist())
  p_hold  = set(df.loc[df["SEASON_YEAR"] == SEASON_HOLDOUT, "PLAYER_ID"].dropna().astype(int).unique().tolist())
  p_ok = np.array(sorted(list(p_train.intersection(p_hold))), dtype=int)

  if p_ok.size == 0:
    raise ValueError("No players overlap between train and holdout season")

  rng = np.random.RandomState(42)
  chosen = rng.choice(p_ok, size=min(N_PLAYERS_FAST, p_ok.size), replace=False)
  df = df[df["PLAYER_ID"].astype(int).isin(chosen)].copy()
  gc.collect()

  print(f"Subsampled players: {df['PLAYER_ID'].nunique()} (target={N_PLAYERS_FAST})")

print("Building splits...")
df_train_full, df_cal_full, df_bt_full = make_splits(df)
print(f"Train rows: {len(df_train_full):,}")
print(f"Cal rows:   {len(df_cal_full):,}")
print(f"BT rows:    {len(df_bt_full):,}")

# Leak check
overlap = set(build_game_keys(df_cal_full)).intersection(set(build_game_keys(df_bt_full)))
print(f"Leak check (cal ∩ bt game keys): {len(overlap)} (should be 0)")

ALL_TARGETS = [f"TARGET_{s}" for s in stat_utils.STAT_COLS]

# Clean old outputs (optional)
if os.path.exists(OUT_RESULTS):
  os.remove(OUT_RESULTS)
if os.path.exists(OUT_BUCKETS):
  os.remove(OUT_BUCKETS)

print("\nPer-stat valid-row audit:")
for s in stat_utils.STAT_COLS:
  y = f"TARGET_{s}"
  pl = f"PL_{s}"
  tr = int((df_train_full[pl].notna() & df_train_full[y].notna()).sum())
  ca = int((df_cal_full[pl].notna() & df_cal_full[y].notna()).sum())
  bt = int((df_bt_full[pl].notna() & df_bt_full[y].notna()).sum())
  print(s, "train_valid", tr, "cal_valid", ca, "bt_valid", bt)

def safe_pos_rate(y):
  y64 = np.asarray(y, dtype=np.float64)
  p = float(np.mean(y64))
  if not np.isfinite(p):
    return np.nan
  return min(max(p, 1e-6), 1.0 - 1e-6)

# Calibration helper (only sigmoid_all + sigmoid_tight)
def try_calibrate(base_model, method, Xc, yc):
  yc64 = np.asarray(yc, dtype=np.float64)
  if yc64.size < 50:
    return None, "CAL_TOO_FEW_ROWS"
  if np.unique(yc64).size < 2:
    return None, "CAL_ONE_CLASS"
  try:
    m = CalibratedClassifierCV(estimator=base_model, method=method, cv="prefit")
    m.fit(Xc, yc)
    return m, "OK"
  except Exception as e:
    return None, f"CAL_EXCEPTION:{type(e).__name__}"

# =====================
# RUN STATS
# =====================
for si, target_stat in enumerate(STATS_TO_RUN):
  t0 = time.time()
  ycol = f"TARGET_{target_stat}"
  plcol = f"PL_{target_stat}"

  print("\n" + "=" * 100)
  print(f"[{si+1}/{len(STATS_TO_RUN)}] STAGE 2 RUN: {target_stat}")
  print("=" * 100)

  train_mask = df_train_full[plcol].notna() & df_train_full[ycol].notna()
  cal_mask   = df_cal_full[plcol].notna() & df_cal_full[ycol].notna()
  bt_mask    = df_bt_full[plcol].notna() & df_bt_full[ycol].notna()

  df_tr = df_train_full[train_mask]
  df_ca = df_cal_full[cal_mask]
  df_bt = df_bt_full[bt_mask]

  n_tr, n_ca, n_bt = len(df_tr), len(df_ca), len(df_bt)
  y_tr = df_tr[ycol].to_numpy()
  y_ca = df_ca[ycol].to_numpy()
  y_bt = df_bt[ycol].to_numpy()

  pos_tr = safe_pos_rate(y_tr) if n_tr > 0 else np.nan
  print(f"Valid rows -> train={n_tr:,} cal={n_ca:,} bt={n_bt:,} | train_pos={pos_tr}")

  # Always log a row if skipped
  if (n_tr < MIN_TRAIN) or (n_ca < MIN_CAL) or (n_bt < MIN_BT) or (not np.isfinite(pos_tr)):
    skip_row = {
      "stat": target_stat,
      "status": "SKIPPED_INSUFFICIENT_OR_BAD_POS",
      "train_rows": int(n_tr),
      "cal_rows": int(n_ca),
      "bt_rows": int(n_bt),
      "train_pos": pos_tr,
    }
    append_rows_to_csv(OUT_RESULTS, [skip_row])
    print("STATUS: SKIPPED (logged)")
    continue

  # Tight masks
  ca_tight_mask = tight_mask_for_stat(df_ca, target_stat)
  bt_tight_mask = tight_mask_for_stat(df_bt, target_stat)
  n_ca_t, n_bt_t = int(ca_tight_mask.sum()), int(bt_tight_mask.sum())
  print(f"Tight rows -> cal_tight={n_ca_t:,} bt_tight={n_bt_t:,}")

  scale_pos = float((1.0 - pos_tr) / pos_tr)

  # feature modes
  for feature_mode in ["full", "reduced"]:
    print(f"\n--- Feature mode: {feature_mode} ---")
    try:
      X_tr = build_feature_frame(df_tr, target_stat, stat_utils.STAT_COLS, ALL_TARGETS, feature_mode=feature_mode)
      X_ca = build_feature_frame(df_ca, target_stat, stat_utils.STAT_COLS, ALL_TARGETS, feature_mode=feature_mode)
      X_bt = build_feature_frame(df_bt, target_stat, stat_utils.STAT_COLS, ALL_TARGETS, feature_mode=feature_mode)

      # Align columns
      for c in X_tr.columns:
        if c not in X_ca.columns:
          X_ca[c] = 0
        if c not in X_bt.columns:
          X_bt[c] = 0
      X_ca = X_ca[X_tr.columns]
      X_bt = X_bt[X_tr.columns]
      n_feat = int(X_tr.shape[1])
      print(f"Features: {n_feat}")

      # arrays (float64 for safety)
      X_tr_np = X_tr.to_numpy(dtype=np.float64, copy=False)
      X_ca_np = X_ca.to_numpy(dtype=np.float64, copy=False)
      X_bt_np = X_bt.to_numpy(dtype=np.float64, copy=False)

      X_ca_t_np = X_ca_np[ca_tight_mask]
      y_ca_t = y_ca[ca_tight_mask]
      X_bt_t_np = X_bt_np[bt_tight_mask]
      y_bt_t = y_bt[bt_tight_mask]

      del X_tr, X_ca, X_bt
      gc.collect()

    except Exception as e:
      fail_row = {
        "stat": target_stat,
        "feature_mode": feature_mode,
        "status": f"FEATURE_BUILD_FAILED:{type(e).__name__}",
        "train_rows": int(n_tr),
        "cal_rows": int(n_ca),
        "bt_rows": int(n_bt),
      }
      append_rows_to_csv(OUT_RESULTS, [fail_row])
      print("FEATURE BUILD FAILED (logged):", type(e).__name__)
      continue

    # HP configs
    for cfg_id, hp in enumerate(HP_GRID):
      print(f"  -> hp_cfg {cfg_id}/{len(HP_GRID)-1}: {hp}")

      params = dict(BASE_XGB_PARAMS)
      params.update(hp)
      params["scale_pos_weight"] = scale_pos

      # Always catch exceptions so we don't lose later stats
      try:
        base_model = xgb.XGBClassifier(**params)
        base_model.fit(
          X_tr_np, y_tr,
          eval_set=[(X_ca_np, y_ca)],
          verbose=False
        )
        best_it = int(getattr(base_model, "best_iteration", -1))
      except Exception as e:
        err_row = {
          "stat": target_stat,
          "feature_mode": feature_mode,
          "hp_cfg_id": cfg_id,
          "calibration": "NA",
          "status": f"TRAIN_FAILED:{type(e).__name__}",
          "n_features": n_feat,
        }
        append_rows_to_csv(OUT_RESULTS, [err_row])
        print("    TRAIN FAILED (logged):", type(e).__name__)
        continue

      # Calibration variants (Stage 2 reduced set)
      models = {
        "raw": (base_model, "OK"),
      }

      cal_all, st_all = try_calibrate(base_model, "sigmoid", X_ca_np, y_ca)
      models["sigmoid_all"] = (cal_all, st_all)

      cal_tight, st_tight = (None, "CAL_TOO_FEW_TIGHT_ROWS")
      if len(y_ca_t) >= MIN_CAL_TIGHT:
        cal_tight, st_tight = try_calibrate(base_model, "sigmoid", X_ca_t_np, y_ca_t)
      models["sigmoid_tight"] = (cal_tight, st_tight)

      rows_to_write = []
      buckets_to_write = []

      for cal_name, (mdl, cal_status) in models.items():
        if mdl is None:
          rows_to_write.append({
            "stat": target_stat,
            "feature_mode": feature_mode,
            "hp_cfg_id": cfg_id,
            "calibration": cal_name,
            "status": cal_status,
            "train_rows": int(n_tr),
            "cal_rows": int(n_ca),
            "cal_tight_rows": int(len(y_ca_t)),
            "bt_rows": int(n_bt),
            "bt_tight_rows": int(len(y_bt_t)),
            "n_features": n_feat,
            "best_iteration": best_it
          })
          continue

        try:
          probs_bt = mdl.predict_proba(X_bt_np)[:, 1]
          probs_bt_t = mdl.predict_proba(X_bt_t_np)[:, 1] if X_bt_t_np.shape[0] > 0 else np.array([])

          if not np.isfinite(probs_bt).all():
            rows_to_write.append({
              "stat": target_stat,
              "feature_mode": feature_mode,
              "hp_cfg_id": cfg_id,
              "calibration": cal_name,
              "status": "BAD_PROBS_NONFINITE",
              "n_features": n_feat,
              "best_iteration": best_it
            })
            continue

          m_full = compute_metrics(probs_bt, y_bt)
          m_tight = compute_metrics(probs_bt_t, y_bt_t) if probs_bt_t.size > 0 else None

          if m_full is None:
            rows_to_write.append({
              "stat": target_stat,
              "feature_mode": feature_mode,
              "hp_cfg_id": cfg_id,
              "calibration": cal_name,
              "status": "METRICS_FAILED",
              "n_features": n_feat,
              "best_iteration": best_it
            })
            continue

          out = {
            "stat": target_stat,
            "feature_mode": feature_mode,
            "hp_cfg_id": cfg_id,
            "calibration": cal_name,
            "status": "OK",
            "train_rows": int(n_tr),
            "cal_rows": int(n_ca),
            "cal_tight_rows": int(len(y_ca_t)),
            "bt_rows": int(n_bt),
            "bt_tight_rows": int(len(y_bt_t)),
            "n_features": n_feat,
            "best_iteration": best_it
          }

          for k, v in m_full.items():
            out[f"bt_full_{k}"] = v

          if m_tight is not None:
            for k, v in m_tight.items():
              out[f"bt_tight_{k}"] = v
          else:
            for k in m_full.keys():
              out[f"bt_tight_{k}"] = np.nan

          rows_to_write.append(out)

          # lightweight bucket sanity
          if probs_bt_t.size > 0 and np.isfinite(probs_bt_t).all():
            rep = bucket_report_full(probs_bt_t, y_bt_t)
            buckets_to_write.append({
              "stat": target_stat,
              "feature_mode": feature_mode,
              "hp_cfg_id": cfg_id,
              "calibration": cal_name,
              "bt_tight_rows": int(len(y_bt_t)),
              **rep
            })

        except Exception as e:
          rows_to_write.append({
            "stat": target_stat,
            "feature_mode": feature_mode,
            "hp_cfg_id": cfg_id,
            "calibration": cal_name,
            "status": f"EVAL_FAILED:{type(e).__name__}",
            "n_features": n_feat,
            "best_iteration": best_it
          })

      # checkpoint append
      append_rows_to_csv(OUT_RESULTS, rows_to_write)
      append_rows_to_csv(OUT_BUCKETS, buckets_to_write)

      del base_model
      gc.collect()

    # cleanup arrays per feature_mode
    del X_tr_np, X_ca_np, X_bt_np, X_ca_t_np, X_bt_t_np
    gc.collect()

  # cleanup per stat
  del df_tr, df_ca, df_bt, y_tr, y_ca, y_bt
  gc.collect()

  print(f"Completed {target_stat} in {(time.time()-t0)/60:.1f} min")
  print(f"Checkpoint written -> {OUT_RESULTS}, {OUT_BUCKETS}")

print("\nDONE. Outputs:")
print(" -", OUT_RESULTS)
print(" -", OUT_BUCKETS)

!zip -q stage2_outputs.zip stage2_tuning_results.csv stage2_bucket_sanity.csv
files.download("stage2_outputs.zip")

Running stats: ['PTS', 'REB', 'AST', 'STL', 'BLK', 'PRA', 'PA', 'PR', 'RA', 'SB', 'TOV', 'FTA', 'FTM', 'FGA', 'FGM', '3PM', '3PA']
Loading data...
Before optimization: 7.16 GB
After optimization: 2.70 GB
  Float16 columns: 325
  Float32 columns: 135
Subsampled players: 45 (target=45)
Building splits...
Train rows: 585,466
Cal rows:   35,567
BT rows:    15,841
Leak check (cal ∩ bt game keys): 0 (should be 0)

Per-stat valid-row audit:
PTS train_valid 471367 cal_valid 28716 bt_valid 13057
REB train_valid 200343 cal_valid 11474 bt_valid 5107
AST train_valid 148766 cal_valid 9760 bt_valid 4139
STL train_valid 69420 cal_valid 4333 bt_valid 1953
BLK train_valid 55661 cal_valid 3244 bt_valid 1388
PRA train_valid 570773 cal_valid 34322 bt_valid 15580
PA train_valid 503685 cal_valid 30544 bt_valid 13854
PR train_valid 541470 cal_valid 32421 bt_valid 14738
RA train_valid 272910 cal_valid 16309 bt_valid 7283
SB train_valid 98146 cal_valid 5685 bt_valid 2642
TOV train_valid 104116 cal_valid 6341 b

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np

RES_PATH = "stage2_tuning_results.csv"
BUC_PATH = "stage2_bucket_sanity.csv"

KEY = ["stat", "feature_mode", "hp_cfg_id", "calibration"]

res = pd.read_csv(RES_PATH)
buck = pd.read_csv(BUC_PATH)

print("Rows:", len(res), "tuning |", len(buck), "bucket")
print("Expected rows (17*2*3*3) = 306")
print("\nStatus counts:")
print(res["status"].value_counts(dropna=False))

# Merge ONLY confidence summaries to avoid bt_tight_rows name collisions
df = res.merge(
    buck[KEY + ["p99_conf", "max_conf"]],
    on=KEY,
    how="left",
    validate="one_to_one"
)

# -----------------------
# Best row per stat (tight logloss, then brier, then ece, then higher auc)
# -----------------------
sort_cols = ["bt_tight_logloss", "bt_tight_brier", "bt_tight_ece10", "bt_tight_auc"]
ascending = [True, True, True, False]

best_by_stat = (
    df.sort_values(sort_cols, ascending=ascending)
      .groupby("stat", as_index=False)
      .head(1)
      .reset_index(drop=True)
)

show_cols = [
    "stat", "feature_mode", "hp_cfg_id", "calibration",
    "bt_tight_rows",
    "bt_tight_logloss", "bt_tight_auc", "bt_tight_ece10",
    "bt_tight_acc@0.60", "bt_tight_cov@0.60",
    "p99_conf", "max_conf",
]
print("\nBest-by-stat summary (tight):")
print(best_by_stat[show_cols].sort_values("bt_tight_logloss").round(4).to_string(index=False))

print("\nEasiest stats (best tight AUC):")
print(
    best_by_stat[["stat", "bt_tight_auc", "bt_tight_logloss"]]
      .sort_values("bt_tight_auc", ascending=False)
      .head(6)
      .round(4)
      .to_string(index=False)
)

print("\nHardest stats (best tight AUC):")
print(
    best_by_stat[["stat", "bt_tight_auc", "bt_tight_logloss"]]
      .sort_values("bt_tight_auc", ascending=True)
      .head(6)
      .round(4)
      .to_string(index=False)
)

# Threshold tradeoffs (averaged across best-per-stat selections)
for t in ["0.55", "0.60", "0.65", "0.70"]:
    acc = best_by_stat[f"bt_tight_acc@{t}"].mean()
    cov = best_by_stat[f"bt_tight_cov@{t}"].mean()
    print(f"\nBest-per-stat avg (tight) @ {t}: acc={acc:.4f} | cov={cov:.4f}")

# -----------------------
# Config leaderboard across stats
# -----------------------
metrics = [
    "bt_tight_logloss", "bt_tight_brier", "bt_tight_ece10", "bt_tight_auc",
    "bt_tight_acc@0.60", "bt_tight_cov@0.60",
    "bt_full_logloss", "bt_full_brier", "bt_full_ece10", "bt_full_auc",
]

cfg = (
    df.groupby(["feature_mode", "hp_cfg_id", "calibration"], as_index=False)[metrics + ["p99_conf", "max_conf"]]
      .mean()
)

# Composite score: lower is better (tight-focused)
cfg["score_tight"] = (
    cfg["bt_tight_logloss"]
    + cfg["bt_tight_brier"]
    + cfg["bt_tight_ece10"]
    - cfg["bt_tight_auc"]
)

print("\nTop configs (mean across stats, tight-focused):")
print(
    cfg.sort_values("score_tight")
      .head(10)[["feature_mode","hp_cfg_id","calibration","score_tight",
                 "bt_tight_logloss","bt_tight_auc","bt_tight_ece10",
                 "bt_tight_acc@0.60","bt_tight_cov@0.60",
                 "p99_conf","max_conf"]]
      .round(5)
      .to_string(index=False)
)

print("\nWorst configs (mean across stats, tight-focused):")
print(
    cfg.sort_values("score_tight", ascending=False)
      .head(6)[["feature_mode","hp_cfg_id","calibration","score_tight",
                "bt_tight_logloss","bt_tight_auc","bt_tight_ece10"]]
      .round(5)
      .to_string(index=False)
)

# -----------------------
# Calibration uplift (global)
# -----------------------
cal_summary = df.groupby("calibration")[["bt_tight_logloss","bt_tight_ece10","bt_full_logloss","bt_full_ece10"]].mean()
print("\nCalibration uplift (means):")
print(cal_summary.round(6).to_string())

# -----------------------
# Confidence -> accuracy curve for a chosen config (from bucket file)
# -----------------------
CHOSEN = {"feature_mode": "full", "hp_cfg_id": 1, "calibration": "sigmoid_tight"}
sub = buck[
    (buck["feature_mode"] == CHOSEN["feature_mode"]) &
    (buck["hp_cfg_id"] == CHOSEN["hp_cfg_id"]) &
    (buck["calibration"] == CHOSEN["calibration"])
].copy()

bins = ["0.50-0.55","0.55-0.60","0.60-0.65","0.65-0.70","0.70-0.80","0.80-0.90","0.90-1.00"]
rows = []
for b in bins:
    n_col = f"{b}_n"
    acc_col = f"{b}_acc"
    n = sub[n_col].sum()
    acc = np.nan
    if n > 0:
        acc = (sub[n_col] * sub[acc_col]).sum() / n
    rows.append([b, int(n), float(acc) if n > 0 else np.nan])

curve = pd.DataFrame(rows, columns=["conf_bin","n","weighted_acc"])
print("\nConfidence -> accuracy curve for chosen config:", CHOSEN)
print(curve.round(4).to_string(index=False))

# -----------------------
# Save summaries
# -----------------------
best_by_stat.to_csv("stage2_best_by_stat.csv", index=False)
cfg.sort_values("score_tight").to_csv("stage2_config_leaderboard.csv", index=False)
print("\nWrote: stage2_best_by_stat.csv, stage2_config_leaderboard.csv")

Rows: 306 tuning | 306 bucket
Expected rows (17*2*3*3) = 306

Status counts:
status
OK    306
Name: count, dtype: int64

Best-by-stat summary (tight):
stat feature_mode  hp_cfg_id   calibration  bt_tight_rows  bt_tight_logloss  bt_tight_auc  bt_tight_ece10  bt_tight_acc@0.60  bt_tight_cov@0.60  p99_conf  max_conf
 BLK      reduced          0   sigmoid_all           1260            0.5169        0.7371          0.0318             0.7964             0.7873    0.9261    0.9332
 3PM         full          0 sigmoid_tight           1941            0.5766        0.7057          0.0739             0.7733             0.6569    0.9258    0.9294
 STL         full          1 sigmoid_tight           1754            0.5947        0.7045          0.0376             0.7370             0.7976    0.9062    0.9201
 TOV      reduced          1 sigmoid_tight           2134            0.6103        0.6901          0.0173             0.7187             0.6931    0.8672    0.8968
  SB      reduced          1 

In [ ]:
# =========================
# STAGE 4: BUILD 4 XGB CANDIDATES (FULL/REDUCED) x (CAL ALL/TIGHT)
#  - Train: pre-2025-26
#  - Tune/EarlyStop + Calibrate: early chunk of 2025-26
#  - Evaluate: late chunk of 2025-26
#  - Memory optimized: float16/float32 storage, float32 matrices for XGB
#  - Outputs a /models zip with requested folder structure
# =========================

import os, gc, json, time, warnings, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score
from google.colab import files
import joblib

import stat_utils  # expects stat_utils.STAT_COLS
# apply_category_mappings must exist in your notebook.

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.calibration")
warnings.filterwarnings("ignore", category=RuntimeWarning)

# -------------------------
# CONFIG (FROZEN)
# -------------------------
SEASON_HOLDOUT = "2025-26"
CAL_FRAC = 0.70

STATS_TO_RUN = list(stat_utils.STAT_COLS)
CONF_THRESHOLDS = [0.55, 0.60, 0.65, 0.70]

TIGHT_ALPHA = 0.25
TIGHT_FLOOR_DEFAULT = 1.0
TIGHT_FLOOR_BY_STAT = {"PTS": 1.5, "PRA": 1.5, "PA": 1.5, "PR": 1.5, "RA": 1.5}

N_PLAYERS_FAST = None  # set 45 for quick tests

# Freeze hp_cfg_id = 1 (balanced)
HP_FIXED = {"max_depth": 4, "min_child_weight": 3, "subsample": 0.7,
            "colsample_bytree": 0.6, "reg_alpha": 0.1, "gamma": 0.1}

BASE_XGB_PARAMS = {
  "objective": "binary:logistic",
  "eval_metric": "logloss",
  "n_estimators": 1200,
  "learning_rate": 0.02,
  "reg_lambda": 1.0,
  "early_stopping_rounds": 50,
  "n_jobs": -1,
  "random_state": 42,
  "tree_method": "hist",
  "verbosity": 0,
}

# Minimum thresholds
MIN_TRAIN = 1000
MIN_CAL = 250
MIN_BT = 250
MIN_CAL_TIGHT = 200

# Outputs
OUT_RESULTS = "stage4_results.csv"
OUT_BUCKETS = "stage4_bucket_sanity.csv"

# Save package
ROOT_DIR = Path("models")  # <-- root folder to zip+download
MODEL_COMPRESS = 3  # joblib compression (smaller zip)

# -------------------------
# GPU detect (safe)
# -------------------------
def has_gpu():
  try:
    subprocess.check_output(["nvidia-smi"], stderr=subprocess.STDOUT)
    return True
  except Exception:
    return False

GPU_OK = has_gpu()
if GPU_OK:
  try:
    _ = xgb.XGBClassifier(tree_method="hist", device="cuda")
    BASE_XGB_PARAMS["device"] = "cuda"
    print("GPU enabled via device=cuda.")
  except TypeError:
    print("XGBoost does not support device=..., using CPU hist.")
else:
  print("No GPU detected, using CPU hist.")

# -------------------------
# MEMORY OPTIMIZATION
# -------------------------
def optimize_dtypes(df):
  print(f"Before optimization: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
  for col in df.columns:
    if col.startswith("TARGET_"):
      df[col] = df[col].astype(np.float32, copy=False)
      continue

    col_type = df[col].dtype
    if col_type == "object" or col_type.name == "category":
      continue

    if col_type == "int64":
      c_min, c_max = df[col].min(), df[col].max()
      if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
        df[col] = df[col].astype(np.int8)
      elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
        df[col] = df[col].astype(np.int16)
      elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
        df[col] = df[col].astype(np.int32)

    elif col_type == "float64":
      c_min, c_max = df[col].min(), df[col].max()
      if pd.notna(c_min) and pd.notna(c_max) and abs(c_min) < 65000 and abs(c_max) < 65000:
        test_val = df[col].iloc[0] if len(df[col]) > 0 else 0
        if pd.notna(test_val):
          precision_loss = abs(test_val - np.float16(test_val)) / (abs(test_val) + 1e-10)
          if precision_loss < 0.01:
            df[col] = df[col].astype(np.float16)
            continue
      df[col] = df[col].astype(np.float32)

  print(f"After optimization: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
  print(f"  Float16 columns: {len(df.select_dtypes(include=['float16']).columns)}")
  print(f"  Float32 columns: {len(df.select_dtypes(include=['float32']).columns)}")
  return df

# -------------------------
# SPLITS (game-level, chronological)
# -------------------------
def build_game_keys(df):
  return list(zip(df["PLAYER_ID"].astype(int).values, df["GAME_ID"].astype(str).values))

def make_splits(df, season_holdout=SEASON_HOLDOUT, cal_frac=CAL_FRAC):
  train_mask = df["SEASON_YEAR"] != season_holdout
  df_train = df[train_mask]
  df_hold = df[~train_mask].copy()

  if len(df_hold) == 0:
    raise ValueError(f"No rows found for holdout season {season_holdout}")

  g = df_hold.groupby(["PLAYER_ID", "GAME_ID"], sort=False)["GAME_DATE"].min().reset_index()
  g = g.sort_values(["GAME_DATE", "PLAYER_ID", "GAME_ID"]).reset_index(drop=True)

  n_games = len(g)
  cut = int(np.floor(cal_frac * n_games))
  cut = max(1, min(cut, n_games - 1))

  cal_games = set(zip(g.iloc[:cut]["PLAYER_ID"].astype(int), g.iloc[:cut]["GAME_ID"].astype(str)))
  bt_games  = set(zip(g.iloc[cut:]["PLAYER_ID"].astype(int), g.iloc[cut:]["GAME_ID"].astype(str)))

  hold_keys = build_game_keys(df_hold)
  hold_in_cal = np.array([k in cal_games for k in hold_keys], dtype=bool)
  hold_in_bt  = np.array([k in bt_games  for k in hold_keys], dtype=bool)

  df_cal = df_hold.iloc[np.where(hold_in_cal)[0]]
  df_bt  = df_hold.iloc[np.where(hold_in_bt)[0]]
  return df_train, df_cal, df_bt

def tight_mask_for_stat(df, stat, alpha=TIGHT_ALPHA):
  dist = df[f"{stat}_DIST_FROM_ANCHOR"].to_numpy()
  std5 = df[f"STD_L5_AVG_{stat}"].to_numpy()
  stdc = df[f"STD_CUM_AVG_{stat}"].to_numpy()
  std = np.where(np.isfinite(std5), std5, stdc)

  floor = TIGHT_FLOOR_BY_STAT.get(stat, TIGHT_FLOOR_DEFAULT)
  thresh = np.maximum(alpha * std, floor)
  return np.isfinite(dist) & np.isfinite(thresh) & (dist <= thresh)

# -------------------------
# METRICS
# -------------------------
def expected_calibration_error(probs, y_true, n_bins=10):
  probs = np.asarray(probs, dtype=np.float64)
  y_true = np.asarray(y_true, dtype=np.float64)
  m = np.isfinite(probs) & np.isfinite(y_true)
  probs, y_true = probs[m], y_true[m]
  if probs.size == 0:
    return np.nan
  bins = np.linspace(0.0, 1.0, n_bins + 1)
  ece = 0.0
  for i in range(n_bins):
    lo, hi = bins[i], bins[i + 1]
    in_bin = (probs >= lo) & (probs < hi) if i < n_bins - 1 else (probs >= lo) & (probs <= hi)
    if not np.any(in_bin):
      continue
    ece += in_bin.mean() * abs(y_true[in_bin].mean() - probs[in_bin].mean())
  return float(ece)

def compute_metrics(probs, y_true, thresholds=CONF_THRESHOLDS):
  probs = np.asarray(probs, dtype=np.float64)
  y_true = np.asarray(y_true, dtype=np.float64)
  m = np.isfinite(probs) & np.isfinite(y_true)
  probs, y_true = probs[m], y_true[m]
  if probs.size == 0:
    return None

  p_clip = np.clip(probs, 1e-7, 1 - 1e-7)
  out = {}
  out["logloss"] = float(log_loss(y_true, p_clip))
  out["brier"] = float(brier_score_loss(y_true, probs))
  try:
    out["auc"] = float(roc_auc_score(y_true, probs))
  except Exception:
    out["auc"] = np.nan

  out["base_acc"] = float(accuracy_score(y_true, (probs >= 0.5).astype(int)))
  out["ece10"] = expected_calibration_error(probs, y_true, n_bins=10)

  conf = np.maximum(probs, 1.0 - probs)
  out["p99_conf"] = float(np.quantile(conf, 0.99))
  out["max_conf"] = float(conf.max())

  for t in thresholds:
    sel = (probs >= t) | (probs <= (1 - t))
    out[f"cov@{t:.2f}"] = float(sel.mean())
    if sel.sum() == 0:
      out[f"acc@{t:.2f}"] = np.nan
    else:
      out[f"acc@{t:.2f}"] = float(accuracy_score(y_true[sel], (probs[sel] >= 0.5).astype(int)))
  return out

def bucket_report_full(probs, y_true):
  probs = np.asarray(probs, dtype=np.float64)
  y_true = np.asarray(y_true, dtype=np.float64)
  m = np.isfinite(probs) & np.isfinite(y_true)
  probs, y_true = probs[m], y_true[m]
  if probs.size == 0:
    return {}

  conf = np.maximum(probs, 1.0 - probs)
  pred = (probs >= 0.5).astype(int)
  correct = (pred == y_true).astype(int)

  buckets = [
    (0.50, 0.55),(0.55, 0.60),(0.60, 0.65),(0.65, 0.70),
    (0.70, 0.80),(0.80, 0.90),(0.90, 1.00)
  ]
  out = {}
  for lo, hi in buckets:
    key = f"{lo:.2f}-{hi:.2f}"
    in_bin = (conf >= lo) & (conf < hi) if hi < 1.0 else (conf >= lo) & (conf <= hi)
    out[f"{key}_n"] = int(in_bin.sum())
    out[f"{key}_acc"] = float(correct[in_bin].mean()) if in_bin.sum() else np.nan
    out[f"{key}_avg_conf"] = float(conf[in_bin].mean()) if in_bin.sum() else np.nan

  out["p99_conf"] = float(np.quantile(conf, 0.99))
  out["max_conf"] = float(conf.max())
  return out

# -------------------------
# FEATURES
# -------------------------
DROP_BASE_COLS = [
  "PLAYER_NAME", "PLAYER_ID", "TEAM", "MATCHUP", "POSITION",
  "SEASON_YEAR", "SEASON_ID", "GAME_DATE", "GAME_ID",
  "PTS", "REB", "AST", "STL", "BLK", "PRA", "PA", "PR", "RA", "SB",
  "TOV", "FTM", "FGM", "3PM", "FGA", "3PA", "FTA",
  "MIN", "PLUS_MINUS", "TS%", "USG", "OFF_RATING"
]

def build_feature_frame(df, target_stat, stat_cols, all_targets, feature_mode="full"):
  cols_to_drop = list(DROP_BASE_COLS) + list(all_targets)

  for s in stat_cols:
    if s != target_stat:
      cols_to_drop.extend([
        f"PL_{s}",
        f"OVER_PL_RATE_{s}_L10",
        f"OVER_PL_RATE_{s}_L5",
        f"{s}_Z_LINE",
        f"{s}_Z_RECENT",
        f"{s}_Z_MATCHUP",
        f"{s}_LINE_DIFF_X_MIN",
        f"{s}_LINE_DIFF",
        f"{s}_DIST_FROM_ANCHOR",
        f"{s}_ANCHOR"
      ])

  X = df.drop(columns=cols_to_drop, errors="ignore")
  if feature_mode == "full":
    return X

  drop_more = []
  for s in stat_cols:
    if s == target_stat:
      continue

    prefixes = [
      f"CUM_AVG_{s}", f"L5_AVG_{s}", f"STD_CUM_AVG_{s}", f"STD_L5_AVG_{s}",
      f"{s}_",
      f"LAST_MATCHUP_{s}", f"MATCHUP_L4_AVG_{s}", f"MATCHUP_L4_STD_{s}",
      f"CUM_AVG_{s}_PER_MIN", f"L5_{s}_PER_MIN"
    ]
    for p in prefixes:
      drop_more.extend([c for c in X.columns if c.startswith(p)])

    drop_more.extend([c for c in X.columns if c.startswith(f"OPP_ALLOWED_{s}_")])
    drop_more.extend([c for c in X.columns if c.startswith(f"MATCHUP_OPP_ALLOWED_{s}_")])

  if drop_more:
    drop_more = list(dict.fromkeys(drop_more))
    X = X.drop(columns=drop_more, errors="ignore")
  return X

def safe_pos_rate(y):
  y64 = np.asarray(y, dtype=np.float64)
  p = float(np.mean(y64))
  if not np.isfinite(p):
    return np.nan
  return min(max(p, 1e-6), 1.0 - 1e-6)

def try_calibrate(base_model, Xc, yc):
  yc = np.asarray(yc, dtype=np.int8)
  if yc.size < 50:
    return None, "CAL_TOO_FEW_ROWS"
  if np.unique(yc).size < 2:
    return None, "CAL_ONE_CLASS"
  try:
    m = CalibratedClassifierCV(estimator=base_model, method="sigmoid", cv="prefit")
    m.fit(Xc, yc)
    return m, "OK"
  except Exception as e:
    return None, f"CAL_EXCEPTION:{type(e).__name__}"

def append_rows_to_csv(path, rows):
  if not rows:
    return
  df_out = pd.DataFrame(rows)
  write_header = not os.path.exists(path)
  df_out.to_csv(path, mode="a", index=False, header=write_header)

# -------------------------
# OUTPUT FOLDER STRUCTURE
# -------------------------
def champion_name(feature_mode, cal_set):
  if feature_mode == "full" and cal_set == "ALL":
    return "CHAMPION_FULL_ALL"
  if feature_mode == "full" and cal_set == "TIGHT":
    return "CHAMPION_FULL_TIGHT"
  if feature_mode == "reduced" and cal_set == "ALL":
    return "CHAMPION_REDUCED_ALL"
  if feature_mode == "reduced" and cal_set == "TIGHT":
    return "CHAMPION_REDUCED_TIGHT"
  raise ValueError("bad combo")

def ensure_dirs():
  ROOT_DIR.mkdir(parents=True, exist_ok=True)
  for fm in ["full", "reduced"]:
    for cs in ["ALL", "TIGHT"]:
      base = ROOT_DIR / champion_name(fm, cs)
      (base / "models").mkdir(parents=True, exist_ok=True)
      (base / "features").mkdir(parents=True, exist_ok=True)

ensure_dirs()

# -------------------------
# LOAD DATA
# -------------------------
print("Loading data...")
with open("category_mappings.json", "r") as f:
  category_mappings = json.load(f)

df = pd.read_parquet("df_XGB.parquet").replace([np.inf, -np.inf], np.nan)
df = optimize_dtypes(df)

df = apply_category_mappings(df, category_mappings)
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], errors="coerce")

# Optional subsample players (speed)
if N_PLAYERS_FAST is not None:
  p_train = set(df.loc[df["SEASON_YEAR"] != SEASON_HOLDOUT, "PLAYER_ID"].dropna().astype(int).unique().tolist())
  p_hold  = set(df.loc[df["SEASON_YEAR"] == SEASON_HOLDOUT, "PLAYER_ID"].dropna().astype(int).unique().tolist())
  p_ok = np.array(sorted(list(p_train.intersection(p_hold))), dtype=int)
  if p_ok.size == 0:
    raise ValueError("No players overlap between train and holdout season")
  rng = np.random.RandomState(42)
  chosen = rng.choice(p_ok, size=min(N_PLAYERS_FAST, p_ok.size), replace=False)
  df = df[df["PLAYER_ID"].astype(int).isin(chosen)].copy()
  gc.collect()
  print(f"Subsampled players: {df['PLAYER_ID'].nunique()} (target={N_PLAYERS_FAST})")

print("Building splits...")
df_train_full, df_cal_full, df_bt_full = make_splits(df)
print(f"Train rows: {len(df_train_full):,}")
print(f"Cal rows:   {len(df_cal_full):,}")
print(f"BT rows:    {len(df_bt_full):,}")

overlap = set(build_game_keys(df_cal_full)).intersection(set(build_game_keys(df_bt_full)))
print(f"Leak check (cal ∩ bt game keys): {len(overlap)} (should be 0)")

ALL_TARGETS = [f"TARGET_{s}" for s in stat_utils.STAT_COLS]

# Clean metrics outputs
for p in [OUT_RESULTS, OUT_BUCKETS]:
  if os.path.exists(p):
    os.remove(p)

# Feature maps (stat -> list[str])
features_full_map = {}
features_reduced_map = {}

# -------------------------
# RUN STATS
# -------------------------
for si, target_stat in enumerate(STATS_TO_RUN):
  t0 = time.time()
  ycol = f"TARGET_{target_stat}"
  plcol = f"PL_{target_stat}"

  print("\n" + "=" * 100)
  print(f"[{si+1}/{len(STATS_TO_RUN)}] STAGE 4: {target_stat}")
  print("=" * 100)

  tr_mask = df_train_full[plcol].notna() & df_train_full[ycol].notna()
  ca_mask = df_cal_full[plcol].notna() & df_cal_full[ycol].notna()
  bt_mask = df_bt_full[plcol].notna() & df_bt_full[ycol].notna()

  df_tr = df_train_full[tr_mask]
  df_ca = df_cal_full[ca_mask]
  df_bt = df_bt_full[bt_mask]

  n_tr, n_ca, n_bt = len(df_tr), len(df_ca), len(df_bt)
  if n_tr < MIN_TRAIN or n_ca < MIN_CAL or n_bt < MIN_BT:
    print(f"SKIP {target_stat}: insufficient rows train={n_tr}, cal={n_ca}, bt={n_bt}")
    append_rows_to_csv(OUT_RESULTS, [{
      "stat": target_stat, "status": "SKIPPED_INSUFFICIENT",
      "train_rows": int(n_tr), "cal_rows": int(n_ca), "bt_rows": int(n_bt)
    }])
    continue

  y_tr = df_tr[ycol].to_numpy(dtype=np.int8, copy=False)
  y_ca = df_ca[ycol].to_numpy(dtype=np.int8, copy=False)
  y_bt = df_bt[ycol].to_numpy(dtype=np.int8, copy=False)

  pos_tr = safe_pos_rate(y_tr)
  if not np.isfinite(pos_tr):
    print(f"SKIP {target_stat}: bad pos rate")
    append_rows_to_csv(OUT_RESULTS, [{
      "stat": target_stat, "status": "SKIPPED_BAD_POS",
      "train_rows": int(n_tr), "cal_rows": int(n_ca), "bt_rows": int(n_bt),
      "train_pos": pos_tr
    }])
    continue

  scale_pos = float((1.0 - pos_tr) / pos_tr)

  ca_tight = tight_mask_for_stat(df_ca, target_stat)
  bt_tight = tight_mask_for_stat(df_bt, target_stat)

  for feature_mode in ["full", "reduced"]:
    print(f"\n--- feature_mode: {feature_mode} ---")

    X_tr = build_feature_frame(df_tr, target_stat, stat_utils.STAT_COLS, ALL_TARGETS, feature_mode=feature_mode)
    X_ca = build_feature_frame(df_ca, target_stat, stat_utils.STAT_COLS, ALL_TARGETS, feature_mode=feature_mode)
    X_bt = build_feature_frame(df_bt, target_stat, stat_utils.STAT_COLS, ALL_TARGETS, feature_mode=feature_mode)

    for c in X_tr.columns:
      if c not in X_ca.columns:
        X_ca[c] = 0
      if c not in X_bt.columns:
        X_bt[c] = 0
    X_ca = X_ca[X_tr.columns]
    X_bt = X_bt[X_tr.columns]

    cols = list(X_tr.columns)
    if feature_mode == "full":
      features_full_map[target_stat] = cols
    else:
      features_reduced_map[target_stat] = cols

    X_tr_np = X_tr.to_numpy(dtype=np.float32, copy=False)
    X_ca_np = X_ca.to_numpy(dtype=np.float32, copy=False)
    X_bt_np = X_bt.to_numpy(dtype=np.float32, copy=False)

    X_ca_t_np = X_ca_np[ca_tight]
    y_ca_t = y_ca[ca_tight]
    X_bt_t_np = X_bt_np[bt_tight]
    y_bt_t = y_bt[bt_tight]

    del X_tr, X_ca, X_bt
    gc.collect()

    params = dict(BASE_XGB_PARAMS)
    params.update(HP_FIXED)
    params["scale_pos_weight"] = scale_pos

    base_model = xgb.XGBClassifier(**params)
    base_model.fit(X_tr_np, y_tr, eval_set=[(X_ca_np, y_ca)], verbose=False)
    best_it = int(getattr(base_model, "best_iteration", -1))

    cal_variants = [("ALL", X_ca_np, y_ca), ("TIGHT", X_ca_t_np, y_ca_t)]
    rows_out, buckets_out = [], []

    for cal_tag, Xc, yc in cal_variants:
      if cal_tag == "TIGHT" and len(yc) < MIN_CAL_TIGHT:
        rows_out.append({
          "stat": target_stat, "feature_mode": feature_mode, "calibration_set": cal_tag,
          "status": "CAL_TOO_FEW_TIGHT_ROWS",
          "train_rows": int(n_tr), "cal_rows": int(n_ca), "bt_rows": int(n_bt),
          "cal_tight_rows": int(len(y_ca_t)), "bt_tight_rows": int(len(y_bt_t)),
          "n_features": int(X_tr_np.shape[1]), "best_iteration": best_it
        })
        continue

      mdl, cal_status = try_calibrate(base_model, Xc, yc)
      if mdl is None:
        rows_out.append({
          "stat": target_stat, "feature_mode": feature_mode, "calibration_set": cal_tag,
          "status": cal_status,
          "train_rows": int(n_tr), "cal_rows": int(n_ca), "bt_rows": int(n_bt),
          "cal_tight_rows": int(len(y_ca_t)), "bt_tight_rows": int(len(y_bt_t)),
          "n_features": int(X_tr_np.shape[1]), "best_iteration": best_it
        })
        continue

      probs_bt = mdl.predict_proba(X_bt_np)[:, 1]
      probs_bt_t = mdl.predict_proba(X_bt_t_np)[:, 1] if X_bt_t_np.shape[0] else np.array([])

      m_full = compute_metrics(probs_bt, y_bt)
      m_tight = compute_metrics(probs_bt_t, y_bt_t) if probs_bt_t.size else None

      out = {
        "stat": target_stat, "feature_mode": feature_mode, "calibration_set": cal_tag,
        "status": "OK",
        "train_rows": int(n_tr), "cal_rows": int(n_ca), "bt_rows": int(n_bt),
        "cal_tight_rows": int(len(y_ca_t)), "bt_tight_rows": int(len(y_bt_t)),
        "n_features": int(X_tr_np.shape[1]), "best_iteration": best_it,
        "scale_pos_weight": scale_pos,
        **{f"hp_{k}": v for k, v in HP_FIXED.items()},
      }
      for k, v in m_full.items():
        out[f"bt_full_{k}"] = v
      if m_tight is not None:
        for k, v in m_tight.items():
          out[f"bt_tight_{k}"] = v
      rows_out.append(out)

      if probs_bt_t.size:
        rep = bucket_report_full(probs_bt_t, y_bt_t)
        buckets_out.append({
          "stat": target_stat, "feature_mode": feature_mode, "calibration_set": cal_tag,
          "bt_tight_rows": int(len(y_bt_t)), **rep
        })

      champ = champion_name(feature_mode, cal_tag)
      model_dir = ROOT_DIR / champ / "models"
      fname = f"{target_stat}_{feature_mode.upper()}_{cal_tag}.pkl"
      joblib.dump(mdl, model_dir / fname, compress=MODEL_COMPRESS)

    append_rows_to_csv(OUT_RESULTS, rows_out)
    append_rows_to_csv(OUT_BUCKETS, buckets_out)

    del base_model, X_tr_np, X_ca_np, X_bt_np, X_ca_t_np, X_bt_t_np
    gc.collect()

  del df_tr, df_ca, df_bt, y_tr, y_ca, y_bt
  gc.collect()

  print(f"Completed {target_stat} in {(time.time()-t0)/60:.1f} min")

print("\nDONE. Metrics outputs:")
print(" -", OUT_RESULTS)
print(" -", OUT_BUCKETS)

# -------------------------
# SAVE FEATURES (one file per champion folder)
# -------------------------
full_payload = {"type": "feature_map", "feature_mode": "full", "by_stat": features_full_map}
reduced_payload = {"type": "feature_map", "feature_mode": "reduced", "by_stat": features_reduced_map}

joblib.dump(full_payload, ROOT_DIR / "CHAMPION_FULL_ALL" / "features" / "FEATURES_FULL_ALL.pkl", compress=MODEL_COMPRESS)
joblib.dump(full_payload, ROOT_DIR / "CHAMPION_FULL_TIGHT" / "features" / "FEATURES_FULL_TIGHT.pkl", compress=MODEL_COMPRESS)

joblib.dump(reduced_payload, ROOT_DIR / "CHAMPION_REDUCED_ALL" / "features" / "FEATURES_REDUCED_ALL.pkl", compress=MODEL_COMPRESS)
joblib.dump(reduced_payload, ROOT_DIR / "CHAMPION_REDUCED_TIGHT" / "features" / "FEATURES_REDUCED_TIGHT.pkl", compress=MODEL_COMPRESS)

# Save category mappings
with open(ROOT_DIR / "category_mappings.json", "w") as f:
  json.dump(category_mappings, f, indent=2)

# Save metrics CSVs at root too (handy)
pd.read_csv(OUT_RESULTS).to_csv(ROOT_DIR / OUT_RESULTS, index=False)
pd.read_csv(OUT_BUCKETS).to_csv(ROOT_DIR / OUT_BUCKETS, index=False)

# Zip + download
zip_name = "models_package.zip"
!zip -qr {zip_name} models
files.download(zip_name)
print("Downloaded:", zip_name)

GPU enabled via device=cuda.
Loading data...
Before optimization: 6.97 GB
After optimization: 2.83 GB
  Float16 columns: 292
  Float32 columns: 169
Building splits...
Train rows: 1,627,000
Cal rows:   114,775
BT rows:    52,077
Leak check (cal ∩ bt game keys): 0 (should be 0)

[1/17] STAGE 4: PTS

--- feature_mode: full ---

--- feature_mode: reduced ---
Completed PTS in 1.6 min

[2/17] STAGE 4: REB

--- feature_mode: full ---

--- feature_mode: reduced ---
Completed REB in 0.5 min

[3/17] STAGE 4: AST

--- feature_mode: full ---

--- feature_mode: reduced ---
Completed AST in 0.5 min

[4/17] STAGE 4: STL

--- feature_mode: full ---

--- feature_mode: reduced ---
Completed STL in 0.3 min

[5/17] STAGE 4: BLK

--- feature_mode: full ---

--- feature_mode: reduced ---
Completed BLK in 0.3 min

[6/17] STAGE 4: PRA

--- feature_mode: full ---

--- feature_mode: reduced ---
Completed PRA in 1.8 min

[7/17] STAGE 4: PA

--- feature_mode: full ---

--- feature_mode: reduced ---
Completed PA i

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: models_package.zip


In [ ]:
# =========================
# EVALUATE STAGE 4 MODELS (PRINT ACC + THRESHOLD ACC/COV + TOP 15 FEATURES)
#  - Loads models from ./models/CHAMPION_*/models/*.pkl
#  - Loads features map from ./models/CHAMPION_*/features/FEATURES_*.pkl
#  - Rebuilds backtest split (late 2025-26 chunk)
#  - For each stat x each model:
#     * base accuracy over full BT
#     * threshold accuracy + coverage + n
#     * top 15 features by XGB gain
#  - Saves: stage4_eval_accuracy_summary.csv
# =========================

import os, gc, json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import accuracy_score

import stat_utils  # expects stat_utils.STAT_COLS
# apply_category_mappings must exist in your notebook

# -------------------------
# CONFIG (must match training)
# -------------------------
SEASON_HOLDOUT = "2025-26"
CAL_FRAC = 0.70

STAT_COLS = list(stat_utils.STAT_COLS)
ALL_TARGETS = [f"TARGET_{s}" for s in STAT_COLS]
CONF_THRESHOLDS = [0.55, 0.60, 0.65, 0.70]

TIGHT_ALPHA = 0.25
TIGHT_FLOOR_DEFAULT = 1.0
TIGHT_FLOOR_BY_STAT = {"PTS": 1.5, "PRA": 1.5, "PA": 1.5, "PR": 1.5, "RA": 1.5}

MODELS_ROOT = Path("models")

CANDIDATES = [
  {"name": "FULL_ALL",     "folder": "CHAMPION_FULL_ALL",     "feature_mode": "full",    "cal_tag": "ALL",   "features_file": "FEATURES_FULL_ALL.pkl"},
  {"name": "FULL_TIGHT",   "folder": "CHAMPION_FULL_TIGHT",   "feature_mode": "full",    "cal_tag": "TIGHT", "features_file": "FEATURES_FULL_TIGHT.pkl"},
  {"name": "REDUCED_ALL",  "folder": "CHAMPION_REDUCED_ALL",  "feature_mode": "reduced", "cal_tag": "ALL",   "features_file": "FEATURES_REDUCED_ALL.pkl"},
  {"name": "REDUCED_TIGHT","folder": "CHAMPION_REDUCED_TIGHT","feature_mode": "reduced", "cal_tag": "TIGHT", "features_file": "FEATURES_REDUCED_TIGHT.pkl"},
]

# -------------------------
# SPLITS (same as training)
# -------------------------
def build_game_keys(df):
  return list(zip(df["PLAYER_ID"].astype(int).values, df["GAME_ID"].astype(str).values))

def make_splits(df, season_holdout=SEASON_HOLDOUT, cal_frac=CAL_FRAC):
  train_mask = df["SEASON_YEAR"] != season_holdout
  df_train = df[train_mask]
  df_hold = df[~train_mask].copy()

  if len(df_hold) == 0:
    raise ValueError(f"No rows found for holdout season {season_holdout}")

  g = df_hold.groupby(["PLAYER_ID", "GAME_ID"], sort=False)["GAME_DATE"].min().reset_index()
  g = g.sort_values(["GAME_DATE", "PLAYER_ID", "GAME_ID"]).reset_index(drop=True)

  n_games = len(g)
  cut = int(np.floor(cal_frac * n_games))
  cut = max(1, min(cut, n_games - 1))

  cal_games = set(zip(g.iloc[:cut]["PLAYER_ID"].astype(int), g.iloc[:cut]["GAME_ID"].astype(str)))
  bt_games  = set(zip(g.iloc[cut:]["PLAYER_ID"].astype(int), g.iloc[cut:]["GAME_ID"].astype(str)))

  hold_keys = build_game_keys(df_hold)
  hold_in_cal = np.array([k in cal_games for k in hold_keys], dtype=bool)
  hold_in_bt  = np.array([k in bt_games  for k in hold_keys], dtype=bool)

  df_cal = df_hold.iloc[np.where(hold_in_cal)[0]]
  df_bt  = df_hold.iloc[np.where(hold_in_bt)[0]]
  return df_train, df_cal, df_bt

def tight_mask_for_stat(df, stat, alpha=TIGHT_ALPHA):
  dist = df[f"{stat}_DIST_FROM_ANCHOR"].to_numpy()
  std5 = df[f"STD_L5_AVG_{stat}"].to_numpy()
  stdc = df[f"STD_CUM_AVG_{stat}"].to_numpy()
  std = np.where(np.isfinite(std5), std5, stdc)

  floor = TIGHT_FLOOR_BY_STAT.get(stat, TIGHT_FLOOR_DEFAULT)
  thresh = np.maximum(alpha * std, floor)
  return np.isfinite(dist) & np.isfinite(thresh) & (dist <= thresh)

# -------------------------
# FEATURES (must match training build_feature_frame)
# -------------------------
DROP_BASE_COLS = [
  "PLAYER_NAME", "PLAYER_ID", "TEAM", "MATCHUP", "POSITION",
  "SEASON_YEAR", "SEASON_ID", "GAME_DATE", "GAME_ID",
  "PTS", "REB", "AST", "STL", "BLK", "PRA", "PA", "PR", "RA", "SB",
  "TOV", "FTM", "FGM", "3PM", "FGA", "3PA", "FTA",
  "MIN", "PLUS_MINUS", "TS%", "USG", "OFF_RATING"
]

def build_feature_frame(df, target_stat, stat_cols, all_targets, feature_mode="full"):
  cols_to_drop = list(DROP_BASE_COLS) + list(all_targets)

  for s in stat_cols:
    if s != target_stat:
      cols_to_drop.extend([
        f"PL_{s}",
        f"OVER_PL_RATE_{s}_L10",
        f"OVER_PL_RATE_{s}_L5",
        f"{s}_Z_LINE",
        f"{s}_Z_RECENT",
        f"{s}_Z_MATCHUP",
        f"{s}_LINE_DIFF_X_MIN",
        f"{s}_LINE_DIFF",
        f"{s}_DIST_FROM_ANCHOR",
        f"{s}_ANCHOR"
      ])

  X = df.drop(columns=cols_to_drop, errors="ignore")
  if feature_mode == "full":
    return X

  drop_more = []
  for s in stat_cols:
    if s == target_stat:
      continue

    prefixes = [
      f"CUM_AVG_{s}", f"L5_AVG_{s}", f"STD_CUM_AVG_{s}", f"STD_L5_AVG_{s}",
      f"{s}_",
      f"LAST_MATCHUP_{s}", f"MATCHUP_L4_AVG_{s}", f"MATCHUP_L4_STD_{s}",
      f"CUM_AVG_{s}_PER_MIN", f"L5_{s}_PER_MIN"
    ]
    for p in prefixes:
      drop_more.extend([c for c in X.columns if c.startswith(p)])

    drop_more.extend([c for c in X.columns if c.startswith(f"OPP_ALLOWED_{s}_")])
    drop_more.extend([c for c in X.columns if c.startswith(f"MATCHUP_OPP_ALLOWED_{s}_")])

  if drop_more:
    drop_more = list(dict.fromkeys(drop_more))
    X = X.drop(columns=drop_more, errors="ignore")
  return X

# -------------------------
# Helpers: extract XGB base estimator + feature importance
# -------------------------
def get_xgb_base_estimator(calibrated_model):
  """
  calibrated_model is expected to be CalibratedClassifierCV (cv='prefit') saved by joblib.
  Return underlying XGBClassifier used as the estimator.
  """
  # Newer sklearn exposes .estimator
  if hasattr(calibrated_model, "estimator") and calibrated_model.estimator is not None:
    return calibrated_model.estimator

  # Most reliable: calibrated_classifiers_[0].estimator
  if hasattr(calibrated_model, "calibrated_classifiers_") and calibrated_model.calibrated_classifiers_:
    cc0 = calibrated_model.calibrated_classifiers_[0]
    if hasattr(cc0, "estimator"):
      return cc0.estimator

  return None

def top_features_by_gain(xgb_estimator, feature_names, topk=15):
  if xgb_estimator is None:
    return []

  if not hasattr(xgb_estimator, "get_booster"):
    return []

  booster = xgb_estimator.get_booster()
  score = booster.get_score(importance_type="gain")  # dict like {"f0": gain, ...}
  if not score:
    return []

  items = []
  for k, v in score.items():
    name = k
    if isinstance(k, str) and k.startswith("f"):
      try:
        idx = int(k[1:])
        if 0 <= idx < len(feature_names):
          name = feature_names[idx]
      except:
        pass
    items.append((name, float(v)))

  items.sort(key=lambda x: x[1], reverse=True)
  return items[:topk]

def threshold_metrics(probs, y_true, thresholds):
  """
  Returns dict with:
    base_acc
    for each t: acc_t, cov_t, n_t
  """
  probs = np.asarray(probs, dtype=np.float64)
  y_true = np.asarray(y_true, dtype=np.int8)

  m = np.isfinite(probs) & np.isfinite(y_true)
  probs, y_true = probs[m], y_true[m]
  if probs.size == 0:
    return None

  base_acc = float(accuracy_score(y_true, (probs >= 0.5).astype(int)))

  out = {"base_acc": base_acc}
  for t in thresholds:
    sel = (probs >= t) | (probs <= (1 - t))
    cov = float(sel.mean())
    n = int(sel.sum())
    if n == 0:
      acc = np.nan
    else:
      acc = float(accuracy_score(y_true[sel], (probs[sel] >= 0.5).astype(int)))
    out[f"acc@{t:.2f}"] = acc
    out[f"cov@{t:.2f}"] = cov
    out[f"n@{t:.2f}"] = n
  return out

# -------------------------
# LOAD DATA + BUILD BT SPLIT
# -------------------------
print("Loading parquet for evaluation...")
df = pd.read_parquet("df_XGB.parquet").replace([np.inf, -np.inf], np.nan)

cm_path = MODELS_ROOT / "category_mappings.json"
if cm_path.exists():
  with open(cm_path, "r") as f:
    category_mappings = json.load(f)
else:
  with open("category_mappings.json", "r") as f:
    category_mappings = json.load(f)

df = apply_category_mappings(df, category_mappings)
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], errors="coerce")

_, _, df_bt_full = make_splits(df)
print("BT rows (late chunk):", len(df_bt_full))

# -------------------------
# Load features maps once
# -------------------------
features_maps = {}
for cand in CANDIDATES:
  name = cand["name"]
  folder = MODELS_ROOT / cand["folder"]
  feats_path = folder / "features" / cand["features_file"]
  if not feats_path.exists():
    raise FileNotFoundError(f"Missing features file for {name}: {feats_path}")
  feats_payload = joblib.load(feats_path)
  feat_map = feats_payload.get("by_stat", {})
  features_maps[name] = feat_map

# -------------------------
# MAIN PRINTED EVALUATION
# -------------------------
summary_rows = []

for stat in STAT_COLS:
  ycol = f"TARGET_{stat}"
  plcol = f"PL_{stat}"

  mask = df_bt_full[plcol].notna() & df_bt_full[ycol].notna()
  df_bt_stat = df_bt_full[mask]
  n_bt = len(df_bt_stat)

  print("\n" + "=" * 100)
  print(f"STAT: {stat} | BT valid rows: {n_bt:,}")
  print("=" * 100)

  if n_bt < 250:
    print("Skipping stat due to too few backtest rows.")
    continue

  for cand in CANDIDATES:
    cname = cand["name"]
    feature_mode = cand["feature_mode"]
    cal_tag = cand["cal_tag"]
    folder = MODELS_ROOT / cand["folder"]

    model_path = folder / "models" / f"{stat}_{feature_mode.upper()}_{cal_tag}.pkl"
    if not model_path.exists():
      print(f"\n[{cname}] MISSING MODEL: {model_path}")
      summary_rows.append({"stat": stat, "candidate": cname, "status": "MISSING_MODEL"})
      continue

    feat_map = features_maps[cname]
    if stat not in feat_map:
      print(f"\n[{cname}] MISSING FEATURES for stat {stat}")
      summary_rows.append({"stat": stat, "candidate": cname, "status": "MISSING_FEATURES"})
      continue

    saved_cols = list(feat_map[stat])

    # Build features using same rules as training, then align to saved list
    X_bt = build_feature_frame(df_bt_stat, stat, STAT_COLS, ALL_TARGETS, feature_mode=feature_mode)
    for c in saved_cols:
      if c not in X_bt.columns:
        X_bt[c] = 0
    X_bt = X_bt[saved_cols]

    y_bt = df_bt_stat[ycol].to_numpy(dtype=np.int8, copy=False)

    # Predict
    mdl = joblib.load(model_path)
    probs = mdl.predict_proba(X_bt.to_numpy(dtype=np.float32, copy=False))[:, 1]

    tm = threshold_metrics(probs, y_bt, CONF_THRESHOLDS)
    if tm is None:
      print(f"\n[{cname}] No valid predictions.")
      summary_rows.append({"stat": stat, "candidate": cname, "status": "NO_VALID_PREDS"})
      continue

    # Feature importance (top 15 by gain)
    base_xgb = get_xgb_base_estimator(mdl)
    top15 = top_features_by_gain(base_xgb, saved_cols, topk=15)

    print("\n" + "-" * 90)
    print(f"MODEL: {cname}  | feature_mode={feature_mode} | calibration_set={cal_tag} | n_features={len(saved_cols)}")
    print(f"Base accuracy (full BT, threshold=0.5): {tm['base_acc']:.4f}")
    print("Confidence thresholds (full BT):")
    for t in CONF_THRESHOLDS:
      acc_t = tm[f"acc@{t:.2f}"]
      cov_t = tm[f"cov@{t:.2f}"]
      n_t = tm[f"n@{t:.2f}"]
      if np.isnan(acc_t):
        print(f"  t={t:.2f}: acc=nan  cov={cov_t:.4f}  n={n_t}")
      else:
        print(f"  t={t:.2f}: acc={acc_t:.4f}  cov={cov_t:.4f}  n={n_t}")

    print("Top 15 features by gain:")
    if not top15:
      print("  (no importance available)")
    else:
      for i, (fname, gain) in enumerate(top15, 1):
        print(f"  {i:2d}. {fname} | gain={gain:.6f}")

    # Save summary row
    row = {
      "stat": stat,
      "candidate": cname,
      "feature_mode": feature_mode,
      "calibration_set": cal_tag,
      "status": "OK",
      "bt_rows": int(n_bt),
      "n_features": int(len(saved_cols)),
      "base_acc": tm["base_acc"],
    }
    for t in CONF_THRESHOLDS:
      row[f"acc@{t:.2f}"] = tm[f"acc@{t:.2f}"]
      row[f"cov@{t:.2f}"] = tm[f"cov@{t:.2f}"]
      row[f"n@{t:.2f}"] = tm[f"n@{t:.2f}"]
    summary_rows.append(row)

    del X_bt, probs, mdl
    gc.collect()

# Save CSV
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("stage4_eval_accuracy_summary.csv", index=False)
print("\nWrote: stage4_eval_accuracy_summary.csv")

# Optional: print quick leaderboard by mean base_acc
ok = summary_df[summary_df["status"] == "OK"].copy()
if len(ok) > 0:
  leader = ok.groupby("candidate", as_index=False)[["base_acc"] + [f"acc@{t:.2f}" for t in CONF_THRESHOLDS]].mean()
  leader = leader.sort_values("base_acc", ascending=False)
  print("\nLeaderboard (mean across stats):")
  print(leader.round(5).to_string(index=False))

Loading parquet for evaluation...
BT rows (late chunk): 50929

STAT: PTS | BT valid rows: 42,456

------------------------------------------------------------------------------------------
MODEL: FULL_ALL  | feature_mode=full | calibration_set=ALL | n_features=291
Base accuracy (full BT, threshold=0.5): 0.6756
Confidence thresholds (full BT):
  t=0.55: acc=0.6977  cov=0.8721  n=37024
  t=0.60: acc=0.7206  cov=0.7427  n=31530
  t=0.65: acc=0.7451  cov=0.6093  n=25869
  t=0.70: acc=0.7714  cov=0.4644  n=19716
Top 15 features by gain:
   1. PTS_Z_RECENT | gain=7058.454590
   2. PTS_Z_LINE | gain=5840.498047
   3. OVER_PL_RATE_PTS_L10 | gain=2353.965332
   4. PTS_DIST_FROM_ANCHOR | gain=1548.684204
   5. PTS_LINE_DIFF_X_MIN | gain=1001.878906
   6. PTS_MOMENTUM | gain=885.551575
   7. PTS_LINE_DIFF | gain=615.180115
   8. OVER_PL_RATE_PTS_L5 | gain=507.911621
   9. PTS_MOMENTUM_X_VOL | gain=477.702881
  10. PTS_Z_MATCHUP | gain=387.788849
  11. PL_PTS | gain=320.535095
  12. L5_AVG_TS% | g

In [ ]:
# EVALUATION OF THE MODEL - CALIBRATED ONLY

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import (
  accuracy_score, log_loss, brier_score_loss,
  mean_absolute_error, mean_squared_error, roc_auc_score
)
from sklearn.calibration import calibration_curve

# ================================================================
# HELPER FUNCTIONS
# ================================================================
def accuracy_metrics(predictions, y_true, stat_name, thresholds=(0.55, 0.60, 0.65, 0.70)):
  """
  Comprehensive accuracy metrics for a single stat.

  Args:
    predictions: 1D array of probabilities
    y_true: 1D array of true labels
    stat_name: Name of the stat (for display)
    thresholds: Confidence thresholds to evaluate
  """
  # Remove any NaN values
  valid_mask = ~np.isnan(predictions) & ~np.isnan(y_true)
  predictions = predictions[valid_mask]
  y_true = y_true[valid_mask]

  if len(predictions) == 0:
    print(f"⚠️ No valid predictions for {stat_name}")
    return None

  # Clip predictions to avoid log(0)
  pred_clipped = np.clip(predictions, 1e-7, 1 - 1e-7)

  # Calculate metrics
  ll = log_loss(y_true, pred_clipped)
  brier = brier_score_loss(y_true, predictions)
  mae = mean_absolute_error(y_true, predictions)
  mse = mean_squared_error(y_true, predictions)
  base_acc = accuracy_score(y_true, (predictions > 0.5).astype(int))

  try:
    auc = roc_auc_score(y_true, predictions)
  except:
    auc = 0.0

  print(f"\n📊 Metrics for {stat_name}:")
  print(f"  LogLoss:     {ll:.4f}")
  print(f"  Brier Score: {brier:.4f}")
  print(f"  AUC:         {auc:.4f}")
  print(f"  MAE:         {mae:.4f}")
  print(f"  MSE:         {mse:.4f}")
  print(f"  Base Acc:    {base_acc:.4f}")

  print(f"\n🎯 Confidence-Based Accuracy:")
  for t in thresholds:
    mask = (predictions >= t) | (predictions <= (1 - t))
    coverage = mask.mean()

    if mask.sum() == 0:
      print(f"  @ {t:.2f}: N/A (Coverage: 0.00%)")
      continue

    bet_preds = (predictions[mask] > 0.5).astype(int)
    bet_true = y_true[mask]
    acc = accuracy_score(bet_true, bet_preds)

    print(f"  @ {t:.2f}: {acc:.4f} (Coverage: {coverage:.2%})")

  return {
    'logloss': ll,
    'brier': brier,
    'auc': auc,
    'base_acc': base_acc
  }

def evaluate_realistic_lines(model, df_val, stat_name, training_features, thresholds=(0.55, 0.60, 0.65, 0.70)):
  """
  Evaluate only on lines near anchor (realistic sportsbook lines).

  Filter: Lines within 0.5 standard deviation of anchor (minimum 1.5)

  Args:
    model: Trained calibrated model
    df_val: Validation dataframe
    stat_name: Name of stat being evaluated
    training_features: List of feature names (in correct order)
    thresholds: Confidence thresholds to evaluate
  """

  # Get player's volatility (standard deviation from last 5 games)
  player_std = df_val[f'STD_L5_AVG_{stat_name}'].values

  # Threshold = max(0.5 * std, 1.5)
  adaptive_threshold = np.maximum(0.25 * player_std, 1.5)

  # Filter for realistic lines using distance from anchor
  abs_dist = df_val[f'{stat_name}_DIST_FROM_ANCHOR'].values

  # Create mask: distance <= threshold (per-row comparison)
  realistic_mask = abs_dist <= adaptive_threshold

  if realistic_mask.sum() < 50:
    print(f"\n  ⚠️ Only {realistic_mask.sum()} realistic lines - skipping realistic evaluation")
    return None

  # Filter data
  df_realistic = df_val[realistic_mask].copy()

  print(f"\n  📊 Realistic samples: {len(df_realistic):,} ({realistic_mask.mean():.1%} of full)")
  print(f"  📏 Avg threshold: ±{adaptive_threshold[realistic_mask].mean():.2f}")
  print(f"  📏 Avg distance: {abs_dist[realistic_mask].mean():.2f}")
  print(f"  📏 Max distance: {abs_dist[realistic_mask].max():.2f}")

  # Prepare features
  X_realistic = df_realistic[training_features].values
  y_realistic = df_realistic[f'TARGET_{stat_name}'].values

  # Predict
  probs = model.predict_proba(X_realistic)[:, 1]

  # Remove any NaN values
  valid_mask = ~np.isnan(probs) & ~np.isnan(y_realistic)
  probs = probs[valid_mask]
  y_realistic = y_realistic[valid_mask]

  if len(probs) == 0:
    print(f"  ⚠️ No valid predictions for realistic lines")
    return None

  # Clip predictions to avoid log(0)
  pred_clipped = np.clip(probs, 1e-7, 1 - 1e-7)

  # Calculate metrics
  ll = log_loss(y_realistic, pred_clipped)
  brier = brier_score_loss(y_realistic, probs)
  mae = mean_absolute_error(y_realistic, probs)
  mse = mean_squared_error(y_realistic, probs)
  base_acc = accuracy_score(y_realistic, (probs > 0.5).astype(int))

  try:
    auc = roc_auc_score(y_realistic, probs)
  except:
    auc = 0.0

  print(f"\n📊 Realistic Line Metrics:")
  print(f"  LogLoss:     {ll:.4f}")
  print(f"  Brier Score: {brier:.4f}")
  print(f"  AUC:         {auc:.4f}")
  print(f"  MAE:         {mae:.4f}")
  print(f"  MSE:         {mse:.4f}")
  print(f"  Base Acc:    {base_acc:.4f}")

  print(f"\n🎯 Realistic Line Confidence-Based Accuracy:")
  for t in thresholds:
    mask = (probs >= t) | (probs <= (1 - t))
    coverage = mask.mean()

    if mask.sum() == 0:
      print(f"  @ {t:.2f}: N/A (Coverage: 0.00%)")
      continue

    bet_preds = (probs[mask] > 0.5).astype(int)
    bet_true = y_realistic[mask]
    acc = accuracy_score(bet_true, bet_preds)

    print(f"  @ {t:.2f}: {acc:.4f} (Coverage: {coverage:.2%})")

  return {
    'realistic_samples': len(df_realistic),
    'realistic_pct': realistic_mask.mean(),
    'realistic_logloss': ll,
    'realistic_brier': brier,
    'realistic_auc': auc,
    'realistic_base_acc': base_acc,
    'realistic_avg_threshold': adaptive_threshold[realistic_mask].mean(),
    'realistic_avg_distance': abs_dist[realistic_mask].mean(),
    'realistic_max_distance': abs_dist[realistic_mask].max(),
  }

# ================================================================
# MAIN EVALUATION LOOP
# ================================================================
ALL_TARGETS = [f'TARGET_{col}' for col in STAT_COLS]
evaluation_results = []

# ================================================================
# LOAD VALIDATION DATA
# ================================================================
if 'df_val_full' not in globals():
  print("🔄 Reloading validation data...")
  df = pd.read_parquet("df_XGB.parquet")
  df = df.replace([np.inf, -np.inf], np.nan)

  with open("category_mappings.json", "r") as f:
    category_mappings = json.load(f)

  df = apply_category_mappings(df, category_mappings)
  df_val_full = df[df['SEASON_YEAR'] == '2025-26'].copy()

  del df
  gc.collect()

  print(f"✅ Validation set: {len(df_val_full):,} rows")

# ================================================================
# EVALUATE EACH STAT
# ================================================================
for stat_name, model_dict in champion_models.items():
  print(f"\n{'#'*80}")
  print(f"# EVALUATING: {stat_name}")
  print(f"{'#'*80}")

  # ================================================================
  # GET CALIBRATED MODEL ONLY
  # ================================================================
  calibrated_model = model_dict.get("calibrated")

  if calibrated_model is None:
    print(f"⚠️ No calibrated model found for {stat_name}")
    continue

  # ================================================================
  # 1. FILTER VALIDATION DATA (MUST MATCH TRAINING!)
  # ================================================================
  val_mask = (
    df_val_full[f'PL_{stat_name}'].notna() &
    df_val_full[f'TARGET_{stat_name}'].notna()
  )

  df_val_filtered = df_val_full[val_mask].copy()

  print(f"\n📊 Validation samples: {len(df_val_filtered):,} ({val_mask.sum()/len(df_val_full)*100:.1f}%)")

  if len(df_val_filtered) < 50:
    print(f"⚠️ Skipping {stat_name} - insufficient validation data")
    continue

  # ================================================================
  # 2. PREPARE FEATURES (MUST MATCH TRAINING!)
  # ================================================================
  cols_to_drop = drop_cols + list(ALL_TARGETS)

  for col in STAT_COLS:
    if col != stat_name:
      cols_to_drop.extend([
        f'PL_{col}',
        f'OVER_PL_RATE_{col}_L10',
        f'OVER_PL_RATE_{col}_L5',
        f'{col}_Z_LINE',
        f'{col}_Z_RECENT',
        f'{col}_Z_MATCHUP',
        f'{col}_LINE_DIFF_X_MIN',
        f'{col}_LINE_DIFF',
        f"{col}_DIST_FROM_ANCHOR",
        f"{col}_ANCHOR"
      ])

  X_val_stat = df_val_filtered.drop(columns=cols_to_drop, errors='ignore')
  y_val_stat = df_val_filtered[f'TARGET_{stat_name}'].values

  # Align features with training
  training_features = feature_names_by_stat[stat_name]

  # Add missing columns
  for col in training_features:
    if col not in X_val_stat.columns:
      X_val_stat[col] = 0

  # Keep only training features in correct order
  X_val_stat = X_val_stat[training_features]

  print(f"  Features: {X_val_stat.shape[1]}")
  print(f"  Positive rate: {y_val_stat.mean():.1%}")

  # ================================================================
  # 3. EVALUATE CALIBRATED MODEL - ALL VALIDATION LINES
  # ================================================================
  print(f"\n{'─'*60}")
  print("🔹 CALIBRATED MODEL - ALL VALIDATION LINES")
  print(f"{'─'*60}")

  cal_probs = calibrated_model.predict_proba(X_val_stat.values)[:, 1]

  full_metrics = accuracy_metrics(
    predictions=cal_probs,
    y_true=y_val_stat,
    stat_name=stat_name
  )

  # ================================================================
  # 4. EVALUATE CALIBRATED MODEL - REALISTIC LINES ONLY
  # ================================================================
  print(f"\n{'─'*60}")
  print("🎯 CALIBRATED MODEL - REALISTIC LINES (Within 0.5σ, min 1.5)")
  print(f"{'─'*60}")

  realistic_metrics = evaluate_realistic_lines(
    model=calibrated_model,
    df_val=df_val_filtered,
    stat_name=stat_name,
    training_features=training_features
  )

  # ================================================================
  # 5. FEATURE IMPORTANCE
  # ================================================================
  print(f"\n{'─'*60}")
  print("🔝 FEATURE IMPORTANCE")
  print(f"{'─'*60}")

  try:
    # Get the underlying XGBoost model from calibrated wrapper
    if hasattr(calibrated_model, 'estimator'):
      base_model = calibrated_model.estimator
    elif hasattr(calibrated_model, 'calibrated_classifiers_'):
      base_model = calibrated_model.calibrated_classifiers_[0].estimator
    else:
      base_model = calibrated_model

    booster = base_model.get_booster()
    importance_dict = booster.get_score(importance_type="gain")

    if len(importance_dict) == 0:
      print("⚠️ No feature importance available")
    else:
      # Map feature indices to names
      feature_importance = []
      for feat_idx, gain in importance_dict.items():
        idx = int(feat_idx[1:])  # Remove 'f' prefix
        if idx < len(training_features):
          feature_importance.append({
            'feature': training_features[idx],
            'gain': gain
          })

      # Sort by importance
      feature_importance = sorted(
        feature_importance,
        key=lambda x: x['gain'],
        reverse=True
      )[:20]

      # Print top 10
      print("\nTop 10 Features:")
      for i, item in enumerate(feature_importance[:10], 1):
        print(f"  {i:2d}. {item['feature']:45s} {item['gain']:8.1f}")

  except Exception as e:
    print(f"⚠️ Error extracting feature importance: {e}")

  # ================================================================
  # 6. STORE RESULTS
  # ================================================================
  if full_metrics:
    result = {
      'Stat': stat_name,
      'Samples': len(df_val_filtered),
      'Pos_Rate': y_val_stat.mean(),
      'full_logloss': full_metrics['logloss'],
      'full_brier': full_metrics['brier'],
      'full_auc': full_metrics['auc'],
      'full_base_acc': full_metrics['base_acc'],
    }

    # Add realistic metrics if available
    if realistic_metrics:
      result.update(realistic_metrics)

    evaluation_results.append(result)

# ================================================================
# 7. SUMMARY
# ================================================================
print(f"\n\n{'#'*80}")
print("# EVALUATION SUMMARY - CALIBRATED MODELS")
print(f"{'#'*80}\n")

if evaluation_results:
  results_df = pd.DataFrame(evaluation_results)
  print(results_df.to_string(index=False))

  # Save
  results_df.to_csv('calibrated_model_evaluation.csv', index=False)
  print("\n✅ Saved: calibrated_model_evaluation.csv")

  # ================================================================
  # 8. COMPARISON: ALL LINES vs REALISTIC LINES
  # ================================================================
  print(f"\n\n{'#'*80}")
  print("# COMPARISON: ALL VALIDATION LINES vs REALISTIC LINES")
  print(f"{'#'*80}\n")

  # Check if realistic metrics exist
  if 'realistic_logloss' in results_df.columns:
    comparison = results_df[[
      'Stat',
      'Samples',
      'full_logloss', 'realistic_logloss',
      'full_auc', 'realistic_auc',
      'full_base_acc', 'realistic_base_acc',
      'realistic_samples', 'realistic_pct'
    ]].copy()

    # Calculate differences
    comparison['LogLoss_Diff'] = comparison['realistic_logloss'] - comparison['full_logloss']
    comparison['AUC_Diff'] = comparison['full_auc'] - comparison['realistic_auc']
    comparison['Acc_Diff'] = comparison['full_base_acc'] - comparison['realistic_base_acc']

    # Format percentages
    comparison['realistic_pct'] = comparison['realistic_pct'] * 100
    comparison['full_base_acc'] = comparison['full_base_acc'] * 100
    comparison['realistic_base_acc'] = comparison['realistic_base_acc'] * 100

    # Sort by accuracy difference (biggest drop first)
    comparison = comparison.sort_values('Acc_Diff', ascending=False)

    print(comparison.to_string(index=False))

    # Highlight concerning drops
    print(f"\n⚠️ Stats with >10% accuracy drop on realistic lines:")
    concerning = comparison[comparison['Acc_Diff'] > 10.0]
    if len(concerning) > 0:
      print(concerning[['Stat', 'full_base_acc', 'realistic_base_acc', 'Acc_Diff']].to_string(index=False))
    else:
      print("  ✅ None! All stats maintain accuracy on realistic lines.")

    # Save comparison
    comparison.to_csv('all_vs_realistic_comparison.csv', index=False)
    print("\n✅ Saved: all_vs_realistic_comparison.csv")

    # ================================================================
    # 9. FINAL RECOMMENDATIONS
    # ================================================================
    print(f"\n\n{'#'*80}")
    print("# BETTING RECOMMENDATIONS (Based on Realistic Lines)")
    print(f"{'#'*80}\n")

    # Filter stats with >70% accuracy on realistic lines
    bettable = comparison[comparison['realistic_base_acc'] >= 70.0].copy()
    bettable = bettable.sort_values('realistic_base_acc', ascending=False)

    if len(bettable) > 0:
      print("✅ BETTABLE STATS (≥70% accuracy on realistic lines):\n")
      print(bettable[['Stat', 'realistic_base_acc', 'realistic_pct', 'realistic_samples']].to_string(index=False))

      # Grade stats
      print("\n📊 GRADING:")
      for _, row in bettable.iterrows():
        acc = row['realistic_base_acc']
        stat = row['Stat']

        if acc >= 80:
          grade = "A"
          recommendation = "BET AGGRESSIVELY"
        elif acc >= 75:
          grade = "B+"
          recommendation = "BET MODERATELY"
        elif acc >= 70:
          grade = "B"
          recommendation = "BET CAUTIOUSLY"
        else:
          grade = "C"
          recommendation = "SKIP"

        print(f"  {stat:5s}: {acc:5.1f}% → Grade {grade} ({recommendation})")
    else:
      print("⚠️ No stats meet the 70% accuracy threshold on realistic lines")

  else:
    print("⚠️ No realistic line evaluations were completed.")
else:
  print("⚠️ No evaluation results generated")

print("\n🏆 Evaluation Complete!")

In [ ]:
df = pd.read_csv("df.csv")
random_row_dict = df[df["SEASON_YEAR"] == "2024-25"].sample(n=1).iloc[0].to_dict()
print(json.dumps(random_row_dict, indent=2, ensure_ascii=False))

{
  "PLAYER_NAME": "Shai Gilgeous-Alexander",
  "PLAYER_ID": 1628983,
  "POSITION": "Guard",
  "HEIGHT": 78,
  "WEIGHT": 195,
  "SEASON_YEAR": "2024-25",
  "SEASON_ID": 42024,
  "GAME_DATE": "2025-06-08",
  "MIN": 36,
  "PTS": 34,
  "REB": 5,
  "AST": 8,
  "TOV": 2,
  "STL": 4,
  "BLK": 1,
  "FGA": 21,
  "FGM": 11,
  "3PM": 1,
  "FTA": 12,
  "FTM": 11,
  "PRA": 47,
  "PA": 42,
  "PR": 39,
  "RA": 13,
  "SB": 5,
  "PLUS_MINUS": 5,
  "TS%": 64.68797564687976,
  "USG": 0.7855555555555556,
  "ORtg": 176.8033946251768,
  "TARGET_PTS": 0,
  "TARGET_REB": 0,
  "TARGET_AST": 0,
  "TARGET_STL": 1,
  "TARGET_BLK": 1,
  "TARGET_TOV": 0,
  "TARGET_FTM": 1,
  "TARGET_FGM": 0,
  "TARGET_3PM": 0,
  "TARGET_PRA": 0,
  "TARGET_PA": 0,
  "TARGET_PR": 0,
  "TARGET_RA": 0,
  "TARGET_SB": 1,
  "PL_PTS": 37.0,
  "PL_REB": 6.5,
  "PL_AST": 8.0,
  "PL_STL": 2.0,
  "PL_BLK": 0.5,
  "PL_TOV": 3.5,
  "PL_FTM": 10.0,
  "PL_FGM": 13.0,
  "PL_3PM": 2.5,
  "PL_PRA": 50.5,
  "PL_PA": 44.5,
  "PL_PR": 43.5,
  "PL_RA":

In [ ]:
df = pd.read_parquet("df_XGB.parquet")

positions = df["POSITION"].dropna().unique()
print(positions)

['Center' 'Center-Forward' 'Forward' 'Forward-Guard' 'Guard'
 'Guard-Forward' 'Forward-Center']


In [ ]:
print("required_names = [")
for name in sorted(stat_utils.required_names):
    print(f"  '{name}',")
print("]")

required_names = [
  'Aaron Gordon',
  'Ace Bailey',
  'Alex Caruso',
  'Alex Sarr',
  'Alperen Sengun',
  'Amen Thompson',
  'Andrew Nembhard',
  'Anthony Davis',
  'Anthony Edwards',
  'Ausar Thompson',
  'Austin Reaves',
  'Bam Adebayo',
  'Bennedict Mathurin',
  'Brandin Podziemski',
  'Brandon Ingram',
  'Brandon Miller',
  'CJ McCollum',
  'Cade Cunningham',
  'Cam Thomas',
  'Chet Holmgren',
  'Coby White',
  'Collin Sexton',
  'Cooper Flagg',
  'D'Angelo Russell',
  'Darius Garland',
  'De'Aaron Fox',
  'DeMar DeRozan',
  'Deandre Ayton',
  'Deni Avdija',
  'Derrick White',
  'Desmond Bane',
  'Devin Booker',
  'Dillon Brooks',
  'Domantas Sabonis',
  'Donovan Mitchell',
  'Donte DiVincenzo',
  'Draymond Green',
  'Dyson Daniels',
  'Evan Mobley',
  'Franz Wagner',
  'Giannis Antetokounmpo',
  'Immanuel Quickley',
  'Isaiah Hartenstein',
  'Ivica Zubac',
  'Ja Morant',
  'Jaden McDaniels',
  'Jaime Jaquez Jr.',
  'Jalen Brunson',
  'Jalen Duren',
  'Jalen Green',
  'Jalen Johns